# Fixed-Template Velocity CASAL Tests

This notebook follows the `fixed_template_velocity_CASAL_notebook_plan.md` guide and tests **projection sampling of velocity** for a **fixed Wyckoff chart**.

The working assumption is deliberately local and conservative:

- the space group and Wyckoff template are fixed per graph,
- the gauge/origin and species-preserving assignment are frozen,
- coordinate repair happens through a fixed-template SSVD projection,
- velocity repair happens through the chart Jacobian and its tangent projector.

This is therefore a **fixed-template kinetic CASAL-like notebook**, not a claim of theorem-exact CASAL inside KLDM.


## Gate Map

This notebook keeps the guide order, but groups a few closely related gates into shared execution cells:

- Gate 0: setup, deterministic fixture, fixed-template selection
- Gates 1-3: fixed-template validity, projection, idempotence
- Gates 4-7: Jacobian, tangent projector, zero-DOF handling, mean-free compatibility
- Gates 8-10: local velocity repairs near the fixed template
- Gates 11-12: KLDM score compatibility and one-step reverse repair
- Gates 13-16: synthetic kinetic CASAL, dual placement, reduced/full-space correction, metric choice
- Gates 17-18: late mini-chain and full-chain fixed-template repair
- Gates 19-20: template sensitivity and chart-state score compatibility

Default profile is intentionally laptop-sized: graphs `1,2,3`, bounded template budgets, and late-state KLDM checks rather than unbounded sweeps.


In [1]:
import dataclasses
import math
import os
import random
import sys
import time
import traceback
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'configs').exists():
    ROOT = ROOT.parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from kldmPlus.algorithm10_casal_chart import Algorithm10Config, _decode_lattice_matrix, _encode_lattice_matrix, _k_to_l, _l_to_k
from kldmPlus.algorithm13_fixed_template_velocity_casal import (
    FixedTemplateVelocityConfig,
    apply_full_space_force,
    apply_reduced_space_force,
    build_cartesian_block_metric,
    build_fractional_metric,
    center_velocity,
    compute_template_jacobian,
    finite_difference_jacobian_error,
    graph_velocity_norm,
    kinetic_position_update,
    lift_reduced_chart_velocity,
    materialize_template,
    mean_norm,
    projector_idempotence_error,
    project_to_fixed_template,
    reduced_chart_velocity,
    site_jacobian_block_ranks,
    tangent_project_velocity,
    tangent_projector,
    tangent_residual_after_centering,
    wrap_residual,
)
from kldmPlus.run_sampling_compare import SamplingCompareRunner, set_seed
from kldmPlus.sample_evaluation import evaluate_csp_reconstruction
from kldmPlus.symmetry.frame_bridge import map_standardized_structure_to_vanilla_frame, standardize_structure
from kldmPlus.symmetry import (
    materialize_pcs_state,
    sample_random_free_vars,
    select_requested_template_state,
    select_requested_template_states,
    validate_requested_space_group,
)
from kldmPlus.symmetry.pcs_projection import (
    _build_vanilla_structure,
    _collapse_centering_equivalent_structure,
    _periodic_pairwise_distances,
    refresh_pcs_state_anchor,
    vanilla_structure_to_model_tensors,
)
from kldmPlus.utils.time import iter_sampling_times

CONFIG_PATH = ROOT / 'configs' / 'kldm_plus' / 'mp_20' / 'mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)
COMPARE_CFG = CONFIG['sampling_compare']

PROFILE = os.environ.get('FIXED_TEMPLATE_VELOCITY_PROFILE', 'laptop').strip().lower()

def profile_default(name: str, laptop: str, deep: str | None = None) -> str:
    if name in os.environ:
        return os.environ[name]
    if PROFILE in {'full', 'deep'} and deep is not None:
        return deep
    return laptop


def parse_int_list(text: str) -> list[int]:
    return [int(v.strip()) for v in str(text).split(',') if v.strip()]


def parse_float_list(text: str) -> list[float]:
    return [float(v.strip()) for v in str(text).split(',') if v.strip()]

SAMPLE_SEED = int(profile_default('FIXED_TEMPLATE_VELOCITY_SEED', '7', '7'))
GRAPH_IDS = parse_int_list(profile_default('FIXED_TEMPLATE_VELOCITY_GRAPH_IDS', '1,2,3', '1,2,3'))
CAPTURE_N_STEPS = int(profile_default('FIXED_TEMPLATE_VELOCITY_CAPTURE_N_STEPS', '40', '80'))
CAPTURE_STEP = int(profile_default('FIXED_TEMPLATE_VELOCITY_CAPTURE_STEP', '36', '72'))
CAPTURE_SWEEP_STEPS = parse_int_list(profile_default('FIXED_TEMPLATE_VELOCITY_CAPTURE_SWEEP', '12,24,36,40', '20,40,60,80'))
MINI_CHAIN_STEPS = int(profile_default('FIXED_TEMPLATE_VELOCITY_MINI_CHAIN_STEPS', '12', '20'))
FULL_CHAIN_STEPS = int(profile_default('FIXED_TEMPLATE_VELOCITY_FULL_CHAIN_STEPS', '40', '80'))
LATE_START_STEP = int(profile_default('FIXED_TEMPLATE_VELOCITY_LATE_START_STEP', '32', '64'))
TEMPLATE_TOP_K = int(profile_default('FIXED_TEMPLATE_VELOCITY_TEMPLATE_TOP_K', '3', '5'))
FIXTURE_MAX_TEMPLATES = int(profile_default('FIXED_TEMPLATE_VELOCITY_MAX_TEMPLATES', '12', '20'))
FIXTURE_TEMPLATE_EVAL_LIMIT = int(profile_default('FIXED_TEMPLATE_VELOCITY_TEMPLATE_EVAL_LIMIT', '4', '6'))
FIXTURE_OPT_STEPS = int(profile_default('FIXED_TEMPLATE_VELOCITY_OPT_STEPS', '80', '120'))
FIXTURE_LR = float(profile_default('FIXED_TEMPLATE_VELOCITY_LR', '5e-2', '5e-2'))
ORACLE_W_MODE = str(profile_default('FIXED_TEMPLATE_VELOCITY_ORACLE_W_MODE', '1', '1')).strip().lower() not in {'0', 'false', 'no'}
EPS_PASS = 1.0e-6
FD_EPS_VALUES = parse_float_list(profile_default('FIXED_TEMPLATE_VELOCITY_FD_EPS', '1e-2,1e-3,1e-4', '1e-2,1e-3,1e-4'))
METRIC_MODES = [part.strip() for part in profile_default('FIXED_TEMPLATE_VELOCITY_METRICS', 'fractional,cartesian', 'fractional,cartesian').split(',') if part.strip()]
print(f'profile={PROFILE} graphs={GRAPH_IDS} capture_step={CAPTURE_STEP}/{CAPTURE_N_STEPS} mini_chain_steps={MINI_CHAIN_STEPS} full_chain_steps={FULL_CHAIN_STEPS} oracle_W_mode={ORACLE_W_MODE}')


MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen
profile=laptop graphs=[1, 2, 3] capture_step=36/40 mini_chain_steps=12 full_chain_steps=40 oracle_W_mode=True


In [2]:
set_seed(SAMPLE_SEED)
runner = SamplingCompareRunner(config_path=CONFIG_PATH)
runner.model.eval()
template_prior = runner._ensure_template_prior()
batch = next(iter(runner.loader)).to(runner.device)
ptr = batch.ptr.tolist()

sampler_cfgs = {cfg['name']: dict(cfg) for cfg in COMPARE_CFG['samplers']}
facit_cfg = sampler_cfgs.get('FacitKLDM_PC', COMPARE_CFG['samplers'][0])
algo10_cfg_map = dict(sampler_cfgs['Algorithm10_CASAL_chart'].get('algorithm10', {}))
BASE_ALGO10 = Algorithm10Config.from_mapping(algo10_cfg_map)


def replace_supported_dataclass(obj, **updates):
    supported = {field.name for field in dataclasses.fields(obj)}
    return dataclasses.replace(obj, **{key: value for key, value in updates.items() if key in supported})


BASE_GUIDE_ALGO10 = replace_supported_dataclass(
    BASE_ALGO10,
    debug=False,
    oracle_wyckoff_debug=True,
    return_best_even_if_invalid=True,
    max_templates=FIXTURE_MAX_TEMPLATES,
    template_eval_limit=FIXTURE_TEMPLATE_EVAL_LIMIT,
    ssvd_max_steps=min(int(getattr(BASE_ALGO10, 'ssvd_max_steps', 8)), 6),
    ssvd_random_restarts=0,
)

FIXED_CFG = FixedTemplateVelocityConfig(projector_damping=1.0e-6)

@dataclass
class GraphCase:
    graph_id: int
    graph_idx0: int
    composition: dict[int, int]
    atomic_numbers: torch.Tensor
    gt_frac: torch.Tensor
    gt_lattice: torch.Tensor
    requested_sg: int


@dataclass
class KLDMState:
    f: torch.Tensor
    v: torch.Tensor
    l: torch.Tensor
    k: torch.Tensor
    h: torch.Tensor
    t: float
    dt: float
    graph_idx0: int
    full_state: Optional[dict[str, Any]] = None
    full_times: Optional[Any] = None


@dataclass
class FixedTemplateFixture:
    case: GraphCase
    state: Any
    theta: torch.Tensor
    tau: torch.Tensor
    z_frac: torch.Tensor
    z_k: torch.Tensor
    z_l: torch.Tensor
    assignment: torch.Tensor
    J: torch.Tensor
    template_labels: str
    template_multiplicities: str
    template_dofs: str
    template_total_dof: int
    requested_sg_match: bool
    composition_match: bool
    projection_objective: float
    projection_rank: int | None
    projection_condition: float | None
    config: FixedTemplateVelocityConfig
    chart_frac: torch.Tensor | None = None
    chart_atomic_numbers: torch.Tensor | None = None
    chart_l: torch.Tensor | None = None
    chart_k: torch.Tensor | None = None
    chart_num_atoms: int = 0
    raw_num_atoms: int = 0
    target_representation_name: str = ''
    anchor_representation_name: str = ''
    uses_oracle_chart: bool = False
    oracle_w_payload: dict[str, Any] | None = None


result_tables: dict[str, pd.DataFrame] = {}
_caches: dict[tuple[Any, ...], Any] = {}


def graph_slice(graph_idx0: int) -> tuple[int, int]:
    return int(ptr[graph_idx0]), int(ptr[graph_idx0 + 1])


def composition_counter(values) -> dict[int, int]:
    arr = [int(v) for v in torch.as_tensor(values, dtype=torch.long).reshape(-1).tolist()]
    return dict(sorted(Counter(arr).items()))


def graph_tensors(graph_idx0: int, *, source=None) -> dict[str, Any]:
    start, end = graph_slice(graph_idx0)
    if source is None:
        pos, l, h = batch.pos, batch.l, batch.atomic_numbers
    else:
        if len(source) == 4:
            pos, _v, l, h = source
        else:
            pos, l, h = source
    return {
        'pos': pos[start:end].detach().clone(),
        'l': l[graph_idx0].detach().clone().reshape(-1),
        'h': h[start:end].detach().clone().to(torch.long),
        'sg': int(torch.as_tensor(batch.space_group).reshape(-1)[graph_idx0].item()),
        'num_atoms': int(end - start),
    }


def load_test_graphs(graph_ids=GRAPH_IDS) -> list[GraphCase]:
    out = []
    for graph_id in graph_ids:
        graph_idx0 = int(graph_id) - 1
        g = graph_tensors(graph_idx0)
        out.append(
            GraphCase(
                graph_id=int(graph_id),
                graph_idx0=graph_idx0,
                composition=composition_counter(g['h']),
                atomic_numbers=g['h'],
                gt_frac=g['pos'],
                gt_lattice=g['l'],
                requested_sg=g['sg'],
            )
        )
    return out


GRAPH_CASES = load_test_graphs()
backend_summary = {
    'state_backend': 'KLDM reverse sampler via SamplingCompareRunner/model._prepare_csp_sampling + reverse_exp_step',
    'projection_backend': 'fixed_template_ssvd_project (CASCAL-style split projection surrogate on fixed-template chart)',
    'oracle_w_source': 'oracle structure -> select_requested_template_states -> PCSTemplateState/template extractor -> Phi_W/J_W',
}
print(f'loaded_graphs={[g.graph_id for g in GRAPH_CASES]} requested_sg={[g.requested_sg for g in GRAPH_CASES]}')
print('backend_summary=', backend_summary)


mps:  False
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
dataset_cache action=load dataset=mp_20 split=train reason=fresh path=/workspace/data/mp_20/processed/train
dataset_cache action=from_cache_path:start dataset=mp_20 split=train
dataset_cache action=from_cache_path:done dataset=mp_20 split=train
dataset_cache action=builder_build:start dataset=mp_20 split=train
dataset_cache action=builder_build:done dataset=mp_20 split=train
template_prior_cache_hit path=/workspace/notebooks/.cache/kldmPlus/template_prior/0448db66e575a7bb.pkl records=43
loaded_graphs=[1, 2, 3] requested_sg=[227, 4, 194]
backend_summary= {'state_backend': 'KLDM reverse sampler via SamplingComp

In [3]:
def dataframe_result(name: str, rows: list[dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if 'PASS' not in df.columns:
        df['PASS'] = False
    if 'status' not in df.columns:
        df['status'] = np.where(df['PASS'], 'PASS', 'FAIL')
    result_tables[name] = df
    return df


def error_row(test: str, graph: int | str, exc: Exception, failure_category: str, **extra) -> dict[str, Any]:
    tb = traceback.format_exc().strip().splitlines()
    return {
        'test': test,
        'graph': graph,
        'PASS': False,
        'status': 'ERROR',
        'error_type': type(exc).__name__,
        'error_message': str(exc),
        'traceback_tail': tb[-1] if tb else '',
        'failure_category': failure_category,
        **extra,
    }


def print_group_pass(df: pd.DataFrame, group_cols) -> None:
    if df.empty:
        print('empty')
        return
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    cols = list(group_cols) + ['PASS', 'status']
    display(df[cols].groupby(group_cols, as_index=False).agg({'PASS': 'all', 'status': 'last'}))


def l_to_k(l: torch.Tensor, graph_case: GraphCase) -> torch.Tensor:
    return _l_to_k(
        l=l.reshape(-1),
        num_atoms=int(graph_case.gt_frac.shape[0]),
        lattice_transform=runner.lattice_transform,
    ).to(device=l.device, dtype=l.dtype)


def k_to_l(k: torch.Tensor, graph_case: GraphCase) -> torch.Tensor:
    return _k_to_l(
        k=k.reshape(-1),
        num_atoms=int(graph_case.gt_frac.shape[0]),
        lattice_transform=runner.lattice_transform,
    ).to(device=k.device, dtype=k.dtype)


def ensure_l_feature(l: torch.Tensor, num_atoms: int) -> torch.Tensor:
    flat = l.reshape(-1)
    if int(flat.numel()) == 9:
        encoded = _encode_lattice_matrix(
            cell_matrix=flat.reshape(3, 3),
            num_atoms=int(num_atoms),
            lattice_transform=runner.lattice_transform,
        )
        return encoded.to(device=l.device, dtype=l.dtype).reshape(-1)
    return flat


def cell_from_l(l: torch.Tensor, num_atoms: int) -> torch.Tensor:
    flat = l.reshape(-1)
    if int(flat.numel()) == 9:
        return flat.reshape(3, 3).detach()
    return _decode_lattice_matrix(
        l=flat,
        num_atoms=int(num_atoms),
        lattice_transform=runner.lattice_transform,
    ).detach()


def volume_from_l(l: torch.Tensor, num_atoms: int) -> float:
    return float(torch.abs(torch.linalg.det(cell_from_l(l, num_atoms))).detach().item())


def min_pair_distance_from_arrays(frac: torch.Tensor, l: torch.Tensor, num_atoms: int) -> float:
    cell = cell_from_l(l, num_atoms).to(device=frac.device, dtype=frac.dtype)
    distances = _periodic_pairwise_distances(frac_coords=frac, cell_matrix=cell)
    return float(distances.min().detach().item()) if distances.numel() else float('inf')


def cart_distance_to_projection(frac: torch.Tensor, projection, num_atoms: int) -> float:
    cell = cell_from_l(projection.z_l, num_atoms).to(device=frac.device, dtype=frac.dtype)
    delta = wrap_residual(frac, projection.z_frac) @ cell
    return float(torch.linalg.norm(delta.reshape(-1)).detach().item())


def clone_full_state(full_state: dict[str, Any]) -> dict[str, Any]:
    cloned: dict[str, Any] = {}
    for key, value in full_state.items():
        cloned[key] = value.clone() if torch.is_tensor(value) else value
    return cloned


def _native_reverse_step_full_state(full_state: dict[str, Any], times) -> dict[str, Any]:
    with torch.no_grad():
        preds_curr = full_state['score_network'](
            t=times.now.graph,
            pos=full_state['f_t'],
            v=full_state['v_t'],
            h=full_state['a_t'],
            l=full_state['l_t'],
            node_index=full_state['node_index'],
            edge_node_index=full_state['edge_node_index'],
        )
        score_v = full_state['sampling_tdm'].reconstruct_full_reverse_velocity_score(
            t=times.now.nodes,
            v_t=full_state['v_t'],
            pred_v=preds_curr['v'],
            index=full_state['node_index'],
        )
        full_state['f_t'], full_state['v_t'] = full_state['sampling_tdm'].reverse_exp_step(
            f_t=full_state['f_t'],
            v_t=full_state['v_t'],
            score_v=score_v,
            index=full_state['node_index'],
            dt=times.dt,
        )
        full_state['l_t'] = runner.model._reverse_lattice_sampling_step(
            t=times.now.lattice,
            x_t=full_state['l_t'],
            pred=preds_curr['l'],
            dt=times.dt,
            num_atoms=batch.num_atoms,
        )
    return full_state


def graph_state_from_full(full_state: dict[str, Any], case: GraphCase, times=None) -> KLDMState:
    start, end = graph_slice(case.graph_idx0)
    f = full_state['f_t'][start:end].detach().clone()
    v = full_state['v_t'][start:end].detach().clone()
    l = full_state['l_t'][case.graph_idx0].detach().clone().reshape(-1)
    h = full_state['a_t'][start:end].detach().clone().to(torch.long)
    k = l_to_k(l, case)
    dt = float(times.dt) if times is not None else 1.0 / max(1, CAPTURE_N_STEPS)
    t = float(times.now.graph[case.graph_idx0].detach().reshape(-1)[0].item()) if times is not None else float('nan')
    return KLDMState(f=f, v=v, l=l, k=k, h=h, t=t, dt=dt, graph_idx0=case.graph_idx0, full_state=full_state, full_times=times)


def capture_batch_kldm_state(seed=SAMPLE_SEED, n_steps=CAPTURE_N_STEPS, capture_step=CAPTURE_STEP):
    key = ('capture', int(seed), int(n_steps), int(capture_step))
    if key in _caches:
        return _caches[key]
    set_seed(seed)
    state = runner.model._prepare_csp_sampling(
        batch=batch,
        n_steps=int(n_steps),
        t_start=float(facit_cfg.get('t_start', 1.0)),
        t_final=float(facit_cfg.get('t_final', 1e-3)),
    )
    last_times = None
    for step_idx, times in enumerate(iter_sampling_times(batch=state['batch'], grid=state['sampling_time_grid']), start=1):
        last_times = times
        state = _native_reverse_step_full_state(state, times)
        if step_idx >= int(capture_step):
            break
    _caches[key] = (state, last_times, int(capture_step))
    return _caches[key]

# Canonical state-lifting, evaluation, and score-compatibility helpers live in Cell 5.


In [4]:


def metric_tensor(metric_name: str, l: torch.Tensor, num_atoms: int) -> torch.Tensor | None:
    metric_name = str(metric_name).strip().lower()
    if metric_name == 'cartesian':
        cell = cell_from_l(l, num_atoms).to(device=l.device, dtype=l.dtype)
        return build_cartesian_block_metric(cell, num_atoms=num_atoms)
    return build_fractional_metric(num_atoms=num_atoms, dtype=l.dtype, device=l.device)


def template_labels(template) -> str:
    return ','.join(f"{site.atomic_number}@{site.label}" for site in template.site_templates)


def template_multiplicities(template) -> str:
    return ','.join(str(int(site.multiplicity)) for site in template.site_templates)


def template_dofs(template) -> str:
    return ','.join(str(int(site.dof)) for site in template.site_templates)


def l_to_k_num_atoms(l: torch.Tensor, num_atoms: int) -> torch.Tensor:
    return _l_to_k(
        l=ensure_l_feature(l, int(num_atoms)).reshape(-1),
        num_atoms=int(num_atoms),
        lattice_transform=runner.lattice_transform,
    ).to(device=l.device, dtype=l.dtype)


def k_to_l_num_atoms(k: torch.Tensor, num_atoms: int) -> torch.Tensor:
    return _k_to_l(
        k=k.reshape(-1),
        num_atoms=int(num_atoms),
        lattice_transform=runner.lattice_transform,
    ).to(device=k.device, dtype=k.dtype)


def _pcs_state_chart_target(state0) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, str, str]:
    chart_frac = state0.target_frac if state0.target_frac is not None else state0.anchor_frac
    chart_atomic_numbers = state0.target_atomic_numbers if state0.target_atomic_numbers is not None else state0.anchor_atomic_numbers
    chart_cell = state0.target_cell if state0.target_cell is not None else state0.anchor_cell
    chart_k = state0.target_k if state0.target_k is not None else state0.anchor_k
    if chart_frac is None or chart_atomic_numbers is None or chart_cell is None or chart_k is None:
        raise RuntimeError('PCS state is missing oracle chart tensors.')
    return (
        chart_frac.detach().clone(),
        chart_atomic_numbers.detach().clone().to(torch.long),
        chart_cell.detach().clone(),
        chart_k.detach().clone(),
        str(state0.target_representation_name or 'standardized'),
        str(state0.anchor_representation_name or 'standardized'),
    )


def _oracle_w_payload_from_state(case: GraphCase, state0, *, chart_num_atoms: int, chart_repr: str, anchor_repr: str) -> dict[str, Any]:
    return {
        'space_group': int(case.requested_sg),
        'template_labels': template_labels(state0.template),
        'template_multiplicities': template_multiplicities(state0.template),
        'template_dofs': template_dofs(state0.template),
        'template_total_dof': int(state0.template.total_free_dims),
        'template_num_atoms': int(state0.template.total_atoms),
        'chart_num_atoms': int(chart_num_atoms),
        'raw_num_atoms': int(case.gt_frac.shape[0]),
        'target_representation': chart_repr,
        'anchor_representation': anchor_repr,
        'species_assignment_mode': 'oracle_fixed_template',
        'origin_gauge_mode': 'pcs_anchor',
        'extraction_backend': 'select_requested_template_states -> PCSTemplateState',
        'jacobian_backend': 'compute_template_jacobian(materialize_template)',
        'projection_backend': 'fixed_template_ssvd_project',
        'state_backend': 'KLDM reverse sampler',
    }


def _chart_structure_to_vanilla_tensors(
    case: GraphCase,
    fixture: FixedTemplateFixture,
    *,
    frac_coords: torch.Tensor,
    atomic_numbers: torch.Tensor,
    l: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    cell = cell_from_l(l, int(frac_coords.shape[0])).to(device=frac_coords.device, dtype=frac_coords.dtype)
    structure = _build_vanilla_structure(
        frac_coords=frac_coords,
        atomic_numbers=atomic_numbers,
        cell_matrix=cell,
    )
    standardized_like = structure
    translations = fixture.state.target_centering_translations
    if translations is not None and int(atomic_numbers.shape[0]) != int(case.atomic_numbers.shape[0]):
        try:
            standardized_like = _collapse_centering_equivalent_structure(
                structure=structure,
                translations=translations,
                expected_atomic_numbers=fixture.state.bridge.vanilla_atomic_numbers,
            )
        except Exception:
            standardized_like = structure
    try:
        vanilla = map_standardized_structure_to_vanilla_frame(
            standardized_structure=standardized_like,
            vanilla_reference_structure=fixture.state.bridge.vanilla_structure,
            symprec=fixture.state.bridge.symprec,
            angle_tolerance=fixture.state.bridge.angle_tolerance,
        )
    except Exception:
        vanilla = standardized_like
    return vanilla_structure_to_model_tensors(
        structure=vanilla,
        lattice_transform=runner.lattice_transform,
        device=frac_coords.device,
        dtype=frac_coords.dtype,
    )


def _validate_projection_for_case(case: GraphCase, projection, fixture: FixedTemplateFixture | None = None) -> Any:
    structure = _build_vanilla_structure(
        frac_coords=projection.z_frac,
        atomic_numbers=projection.raw.atomic_numbers_chart,
        cell_matrix=projection.raw.cell,
    )
    expected_atomic_numbers = case.atomic_numbers
    if fixture is not None and ORACLE_W_MODE and fixture.uses_oracle_chart:
        expected_atomic_numbers = fixture.chart_atomic_numbers
    return validate_requested_space_group(
        structure=structure,
        requested_space_group=case.requested_sg,
        expected_atomic_numbers=expected_atomic_numbers,
    )


def build_fixed_template_fixture(case: GraphCase) -> FixedTemplateFixture:
    target_cell = cell_from_l(case.gt_lattice, int(case.gt_frac.shape[0])).to(device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    oracle_structure = _build_vanilla_structure(
        frac_coords=case.gt_frac,
        atomic_numbers=case.atomic_numbers,
        cell_matrix=target_cell,
    )

    def _select_candidates(standardization: str):
        return select_requested_template_states(
            frac_coords=case.gt_frac,
            atomic_numbers=case.atomic_numbers,
            cell_matrix=target_cell,
            space_group_number=case.requested_sg,
            standardization=standardization,
            symprec=1e-2,
            angle_tolerance=5.0,
            max_templates=max(FIXTURE_MAX_TEMPLATES, 16),
            template_eval_limit=max(FIXTURE_TEMPLATE_EVAL_LIMIT, 6),
            optimization_steps=FIXTURE_OPT_STEPS,
            learning_rate=FIXTURE_LR,
            coord_weight=1.0,
            lattice_weight=0.0,
            pairdist_weight=0.0,
            steric_weight=0.0,
            volume_weight=0.0,
            k6_weight=0.0,
            freeze_lattice_free_vars=True,
            quick_templates=False,
            top_k=max(TEMPLATE_TOP_K, 6),
            template_prior=template_prior,
            template_prior_weight=0.0,
            oracle_reference_structure=oracle_structure,
            oracle_fit_structure=oracle_structure,
        )

    candidate_entries = []
    selection_errors = []
    for standardization in ['conventional', 'primitive']:
        try:
            states = _select_candidates(standardization)
        except Exception as exc:
            selection_errors.append((standardization, exc))
            continue
        for state0 in states:
            chart_frac, chart_atomic_numbers, chart_cell, chart_k, target_repr, anchor_repr = _pcs_state_chart_target(state0)
            candidate_entries.append({
                'standardization': standardization,
                'state': state0,
                'chart_frac': chart_frac,
                'chart_atomic_numbers': chart_atomic_numbers,
                'chart_cell': chart_cell,
                'chart_k': chart_k,
                'chart_num_atoms': int(chart_atomic_numbers.shape[0]),
                'target_repr': target_repr,
                'anchor_repr': anchor_repr,
                'same_count': int(chart_atomic_numbers.shape[0]) == int(case.gt_frac.shape[0]),
                'expanded': ('expanded' in target_repr) or ('expanded' in anchor_repr) or int(chart_atomic_numbers.shape[0]) != int(case.gt_frac.shape[0]),
            })

    if not candidate_entries:
        lines = [f"{std}:{type(exc).__name__}:{exc}" for std, exc in selection_errors]
        raise RuntimeError(
            f'No fixed-template candidates could be selected for graph={case.graph_id}. selection_errors={lines}'
        )

    candidate_entries.sort(
        key=lambda item: (
            0 if item['standardization'] == 'conventional' else 1,
            0 if item['same_count'] else 1,
            float(item['state'].objective),
        )
    )

    candidate_cfgs = [
        FIXED_CFG,
        dataclasses.replace(
            FIXED_CFG,
            projection_config=dataclasses.replace(FIXED_CFG.projection_config, use_fixed_assignment=False),
        ),
    ]
    best_payload = None
    errors = []
    for item in candidate_entries:
        tau0 = torch.zeros(1, 3, device=item['chart_frac'].device, dtype=item['chart_frac'].dtype)
        for cfg0 in candidate_cfgs:
            try:
                projection = project_to_fixed_template(
                    f_frac=item['chart_frac'],
                    atomic_numbers=item['chart_atomic_numbers'],
                    template_state=item['state'],
                    target_k=item['chart_k'],
                    tau0=tau0,
                    config=cfg0,
                )
                J = compute_template_jacobian(projection.theta, projection.raw.state, tau=projection.tau)
                fixture = FixedTemplateFixture(
                    case=case,
                    state=projection.raw.state,
                    theta=projection.theta,
                    tau=projection.tau,
                    z_frac=projection.z_frac,
                    z_k=projection.z_k,
                    z_l=projection.z_l,
                    assignment=projection.assignment,
                    J=J,
                    template_labels=template_labels(projection.raw.state.template),
                    template_multiplicities=template_multiplicities(projection.raw.state.template),
                    template_dofs=template_dofs(projection.raw.state.template),
                    template_total_dof=int(projection.raw.state.template.total_free_dims),
                    requested_sg_match=False,
                    composition_match=False,
                    projection_objective=float(projection.objective),
                    projection_rank=int(getattr(projection.raw, 'ssvd_rank', 0)) if getattr(projection.raw, 'ssvd_rank', None) is not None else None,
                    projection_condition=float(getattr(projection.raw, 'ssvd_condition_number', float('nan'))) if getattr(projection.raw, 'ssvd_condition_number', None) is not None else None,
                    config=cfg0,
                    chart_frac=item['chart_frac'].detach().clone(),
                    chart_atomic_numbers=item['chart_atomic_numbers'].detach().clone(),
                    chart_l=ensure_l_feature(item['chart_cell'], item['chart_num_atoms']),
                    chart_k=item['chart_k'].detach().clone(),
                    chart_num_atoms=item['chart_num_atoms'],
                    raw_num_atoms=int(case.gt_frac.shape[0]),
                    target_representation_name=item['target_repr'],
                    anchor_representation_name=item['anchor_repr'],
                    uses_oracle_chart=bool(ORACLE_W_MODE),
                    oracle_w_payload=_oracle_w_payload_from_state(
                        case,
                        projection.raw.state,
                        chart_num_atoms=item['chart_num_atoms'],
                        chart_repr=item['target_repr'],
                        anchor_repr=item['anchor_repr'],
                    ),
                )
                validation = _validate_projection_for_case(case, projection, fixture)
                coord_residual = float(torch.linalg.norm(wrap_residual(item['chart_frac'], projection.z_frac).reshape(-1)).detach().item())
                score = (
                    0 if validation.requested_space_group_match else 1,
                    0 if validation.composition_match else 1,
                    0 if item['standardization'] == 'conventional' else 1,
                    0 if bool(getattr(cfg0.projection_config, 'use_fixed_assignment', False)) else 1,
                    coord_residual,
                    float(projection.objective),
                )
                if best_payload is None or score < best_payload[0]:
                    best_payload = (score, fixture, validation)
            except Exception as exc:
                errors.append(
                    f"std={item['standardization']} chart_atoms={item['chart_num_atoms']} target_repr={item['target_repr']} "
                    f"fixed={bool(getattr(cfg0.projection_config, 'use_fixed_assignment', False))} {type(exc).__name__}:{exc}"
                )

    if best_payload is None:
        candidate_summary = [
            {
                'std': item['standardization'],
                'chart_atoms': item['chart_num_atoms'],
                'target_repr': item['target_repr'],
                'anchor_repr': item['anchor_repr'],
                'template_labels': template_labels(item['state'].template),
            }
            for item in candidate_entries[:8]
        ]
        raise RuntimeError(
            f'Could not build an oracle-W fixed template for graph={case.graph_id}. '
            f'candidate_summary={candidate_summary} errors={errors[:8]}'
        )

    _, fixture, validation = best_payload
    fixture.requested_sg_match = bool(validation.requested_space_group_match)
    fixture.composition_match = bool(validation.composition_match)
    return fixture


def _state_to_fixture_chart(case: GraphCase, fixture: FixedTemplateFixture, state: KLDMState) -> KLDMState:
    if not ORACLE_W_MODE or not fixture.uses_oracle_chart:
        return state
    if int(state.f.shape[0]) == int(fixture.chart_num_atoms) and int(state.h.shape[0]) == int(fixture.chart_num_atoms):
        return state
    cell = cell_from_l(state.l, int(state.f.shape[0])).to(device=state.f.device, dtype=state.f.dtype)
    refreshed_state = refresh_pcs_state_anchor(
        state=fixture.state,
        frac_coords=torch.remainder(state.f, 1.0),
        atomic_numbers=state.h.to(device=state.f.device, dtype=torch.long),
        cell_matrix=cell,
        pairdist_weight=0.0,
        pairdist_bins=32,
        pairdist_max_distance=6.0,
        pairdist_bandwidth=0.15,
        coord_weight=1.0,
        lattice_weight=0.0,
        optimization_steps=max(8, min(int(FIXTURE_OPT_STEPS), 32)),
        learning_rate=float(FIXTURE_LR),
    )
    chart_frac, chart_atomic_numbers, chart_cell, chart_k, target_repr, anchor_repr = _pcs_state_chart_target(refreshed_state)
    if int(chart_atomic_numbers.shape[0]) != int(fixture.chart_num_atoms):
        raise RuntimeError(
            f'oracle chart lift mismatch graph={case.graph_id} expected_chart_atoms={fixture.chart_num_atoms} got={int(chart_atomic_numbers.shape[0])} '
            f'target_repr={target_repr!r} anchor_repr={anchor_repr!r} template_atoms={int(refreshed_state.template.total_atoms)}'
        )
    chart_l = ensure_l_feature(chart_cell, int(chart_frac.shape[0]))
    if int(chart_frac.shape[0]) == int(state.v.shape[0]):
        chart_v = state.v.detach().clone()
    else:
        multiplier = max(1, int(chart_frac.shape[0]) // max(int(state.v.shape[0]), 1))
        chart_v = state.v.repeat(multiplier, 1)
    return KLDMState(
        f=chart_frac.detach().clone(),
        v=chart_v.detach().clone(),
        l=chart_l.detach().clone(),
        k=chart_k.detach().clone(),
        h=chart_atomic_numbers.detach().clone().to(device=state.f.device, dtype=torch.long),
        t=state.t,
        dt=state.dt,
        graph_idx0=state.graph_idx0,
        full_state=state.full_state,
        full_times=state.full_times,
    )

def _maybe_lift_state_to_fixture_chart(case: GraphCase, state: KLDMState) -> KLDMState:
    fixture = globals().get('fixtures', {}).get(case.graph_id)
    if fixture is None:
        return state
    return _state_to_fixture_chart(case, fixture, state)


def gt_state_for_graph(case: GraphCase) -> KLDMState:
    f = case.gt_frac.detach().clone()
    l = case.gt_lattice.detach().clone().reshape(-1)
    h = case.atomic_numbers.detach().clone().to(torch.long)
    k = l_to_k(l, case)
    v = torch.zeros_like(f)
    return _maybe_lift_state_to_fixture_chart(case, KLDMState(f=f, v=v, l=l, k=k, h=h, t=0.0, dt=0.0, graph_idx0=case.graph_idx0))


def random_fractional_state(case: GraphCase, *, seed_offset: int = 0) -> KLDMState:
    generator = torch.Generator(device=case.gt_frac.device).manual_seed(SAMPLE_SEED + int(case.graph_id) + int(seed_offset))
    f = torch.rand(case.gt_frac.shape, generator=generator, device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    l = case.gt_lattice.detach().clone().reshape(-1)
    h = case.atomic_numbers.detach().clone().to(torch.long)
    k = l_to_k(l, case)
    v = torch.zeros_like(f)
    return _maybe_lift_state_to_fixture_chart(case, KLDMState(f=f, v=v, l=l, k=k, h=h, t=float('nan'), dt=1.0 / max(1, CAPTURE_N_STEPS), graph_idx0=case.graph_idx0))


def sample_kldm_state_for_graph_at_step(case: GraphCase, *, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS, seed=SAMPLE_SEED) -> KLDMState:
    full_state, times, _step_idx = capture_batch_kldm_state(seed=seed, n_steps=n_steps, capture_step=capture_step)
    return _maybe_lift_state_to_fixture_chart(case, graph_state_from_full(clone_full_state(full_state), case, times=times))


def evaluate_arrays(case: GraphCase, pred_f: torch.Tensor, pred_l: torch.Tensor, pred_h: torch.Tensor) -> dict[str, Any]:
    fixture = globals().get('fixtures', {}).get(case.graph_id)
    eval_f = pred_f
    eval_l = pred_l
    eval_h = pred_h.to(torch.long)
    if fixture is not None and ORACLE_W_MODE and fixture.uses_oracle_chart and int(eval_h.shape[0]) != int(case.atomic_numbers.shape[0]):
        eval_f, eval_l, eval_h = _chart_structure_to_vanilla_tensors(
            case,
            fixture,
            frac_coords=pred_f,
            atomic_numbers=eval_h,
            l=pred_l,
        )
    pred_l_feature = ensure_l_feature(eval_l, int(eval_f.shape[0]))
    result = evaluate_csp_reconstruction(
        pred_f=eval_f,
        pred_l=pred_l_feature,
        pred_a=eval_h,
        target_f=case.gt_frac,
        target_l=case.gt_lattice,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
        requested_space_group=case.requested_sg,
        validity_cutoff=float(getattr(BASE_ALGO10, 'hard_min_distance', 0.5)),
    )
    standardized_frac_rmse = None
    if result.matcher_diagnostics is not None:
        standardized_frac_rmse = result.matcher_diagnostics.standardized_frac_rmse
    return {
        'match': bool(result.match),
        'valid': bool(result.valid),
        'rmse': result.rmse,
        'frac_rmse': result.frac_rmse,
        'standardized_frac_rmse': standardized_frac_rmse,
        'composition_match': result.composition_match,
        'requested_sg_match': result.requested_space_group_match,
        'min_pair_distance': result.min_pair_distance,
        'volume': result.volume,
        'lattice_lengths_mae': result.lattice_lengths_mae,
        'lattice_angles_mae': result.lattice_angles_mae,
        'validity_reason': result.validity_reason,
    }


def score_norms_from_modified_state(case: GraphCase, base_state: KLDMState, *, f: torch.Tensor, v: torch.Tensor, l: torch.Tensor) -> dict[str, Any]:
    if base_state.full_state is None or base_state.full_times is None:
        return {
            'pred_v_norm': float('nan'),
            'score_v_norm': float('nan'),
            'pred_l_norm': float('nan'),
            'finite_ok': False,
        }
    if int(f.shape[0]) != int(case.gt_frac.shape[0]):
        return {
            'pred_v_norm': float('nan'),
            'score_v_norm': float('nan'),
            'pred_l_norm': float('nan'),
            'finite_ok': False,
        }
    full_state = clone_full_state(base_state.full_state)
    times = base_state.full_times
    start, end = graph_slice(case.graph_idx0)
    full_state['f_t'][start:end] = f.to(device=full_state['f_t'].device, dtype=full_state['f_t'].dtype)
    full_state['v_t'][start:end] = v.to(device=full_state['v_t'].device, dtype=full_state['v_t'].dtype)
    l_feature = ensure_l_feature(l, int(case.gt_frac.shape[0]))
    full_state['l_t'][case.graph_idx0] = l_feature.to(device=full_state['l_t'].device, dtype=full_state['l_t'].dtype)
    with torch.no_grad():
        preds = full_state['score_network'](
            t=times.now.graph,
            pos=full_state['f_t'],
            v=full_state['v_t'],
            h=full_state['a_t'],
            l=full_state['l_t'],
            node_index=full_state['node_index'],
            edge_node_index=full_state['edge_node_index'],
        )
        score_v = full_state['sampling_tdm'].reconstruct_full_reverse_velocity_score(
            t=times.now.nodes,
            v_t=full_state['v_t'],
            pred_v=preds['v'],
            index=full_state['node_index'],
        )
    pred_v_graph = preds['v'][start:end]
    score_v_graph = score_v[start:end]
    pred_l_graph = preds['l'][case.graph_idx0].reshape(-1)
    finite_ok = bool(torch.isfinite(pred_v_graph).all() and torch.isfinite(score_v_graph).all() and torch.isfinite(pred_l_graph).all())
    return {
        'pred_v_norm': float(torch.linalg.norm(pred_v_graph.reshape(-1)).detach().item()),
        'score_v_norm': float(torch.linalg.norm(score_v_graph.reshape(-1)).detach().item()),
        'pred_l_norm': float(torch.linalg.norm(pred_l_graph).detach().item()),
        'finite_ok': finite_ok,
    }


def project_state_to_fixture(
    case: GraphCase,
    fixture: FixedTemplateFixture,
    f_frac: torch.Tensor,
    k: torch.Tensor | None = None,
    *,
    use_self_target: bool = False,
    atomic_numbers: torch.Tensor | None = None,
    l_feature: torch.Tensor | None = None,
):
    if atomic_numbers is None:
        if ORACLE_W_MODE and fixture.uses_oracle_chart and int(f_frac.shape[0]) == int(fixture.chart_num_atoms):
            target_atomic_numbers = fixture.chart_atomic_numbers.detach().clone().to(device=f_frac.device, dtype=torch.long)
        else:
            target_atomic_numbers = case.atomic_numbers.detach().clone().to(device=f_frac.device, dtype=torch.long)
    else:
        target_atomic_numbers = atomic_numbers.detach().clone().to(device=f_frac.device, dtype=torch.long)
    if l_feature is None:
        if ORACLE_W_MODE and fixture.uses_oracle_chart and int(f_frac.shape[0]) == int(fixture.chart_num_atoms):
            base_l = fixture.chart_l.detach().clone()
        elif k is not None:
            base_l = k_to_l_num_atoms(k, int(f_frac.shape[0]))
        else:
            base_l = case.gt_lattice
    else:
        base_l = l_feature
    state_like = KLDMState(
        f=f_frac.detach().clone(),
        v=torch.zeros_like(f_frac),
        l=base_l.detach().clone(),
        k=(fixture.chart_k if k is None else k).detach().clone(),
        h=target_atomic_numbers.detach().clone(),
        t=float('nan'),
        dt=0.0,
        graph_idx0=case.graph_idx0,
    )
    if int(state_like.f.shape[0]) != int(fixture.chart_num_atoms) or int(state_like.h.shape[0]) != int(fixture.chart_num_atoms):
        state_like = _state_to_fixture_chart(case, fixture, state_like)
    target_k = (fixture.chart_k if k is None else state_like.k).detach().clone().to(device=state_like.f.device, dtype=state_like.f.dtype)
    template_state = fixture.state
    if use_self_target:
        target_frac = torch.remainder(state_like.f.detach().clone(), 1.0)
        target_l = state_like.l.detach().clone()
        target_cell = cell_from_l(target_l, int(target_frac.shape[0])).to(device=state_like.f.device, dtype=state_like.f.dtype)
        identity = torch.arange(int(target_frac.shape[0]), device=state_like.f.device, dtype=torch.long)
        template_state = dataclasses.replace(
            template_state,
            target_frac=target_frac,
            target_atomic_numbers=state_like.h.detach().clone(),
            target_cell=target_cell.detach().clone(),
            target_k=target_k.detach().clone(),
            fixed_target_assignment=identity.detach().clone(),
            anchor_frac=target_frac.detach().clone(),
            anchor_atomic_numbers=state_like.h.detach().clone(),
            anchor_cell=target_cell.detach().clone(),
            anchor_k=target_k.detach().clone(),
            anchor_assignment=identity.detach().clone(),
        )
    projection = project_to_fixed_template(
        f_frac=state_like.f,
        atomic_numbers=state_like.h,
        template_state=template_state,
        target_k=target_k,
        tau0=fixture.tau,
        config=fixture.config,
    )
    J = compute_template_jacobian(projection.theta, projection.raw.state, tau=projection.tau)
    return projection, J


def template_distance(
    case: GraphCase,
    fixture: FixedTemplateFixture,
    f_frac: torch.Tensor,
    k: torch.Tensor | None = None,
    *,
    use_self_target: bool = False,
    atomic_numbers: torch.Tensor | None = None,
    l_feature: torch.Tensor | None = None,
):
    projection, J = project_state_to_fixture(
        case,
        fixture,
        f_frac,
        k=k,
        use_self_target=use_self_target,
        atomic_numbers=atomic_numbers,
        l_feature=l_feature,
    )
    target_frac = projection.raw.state.target_frac if projection.raw.state.target_frac is not None else fixture.chart_frac
    dist = float(torch.linalg.norm(wrap_residual(target_frac, projection.z_frac).reshape(-1)).detach().item())
    return dist, projection, J


def one_step_repair(case: GraphCase, fixture: FixedTemplateFixture, state: KLDMState, *, mode: str, metric_name: str = 'fractional', eta: float = 0.05):
    state = _state_to_fixture_chart(case, fixture, state)
    projection, J = project_state_to_fixture(case, fixture, state.f, k=state.k, atomic_numbers=state.h, l_feature=state.l)
    metric = metric_tensor(metric_name, projection.z_l.to(device=state.f.device, dtype=state.f.dtype), int(state.f.shape[0]))
    residual = wrap_residual(state.f, projection.z_frac)
    tangent_velocity, projector, mean_norm_before = tangent_project_velocity(
        state.v,
        J=J,
        metric=metric,
        damping=FIXED_CFG.projector_damping,
        mean_free=FIXED_CFG.mean_free_velocity,
    )
    dist_after_projection, _projection_anchor, _J_anchor = template_distance(
        case,
        fixture,
        projection.z_frac,
        k=projection.z_k,
        use_self_target=True,
        atomic_numbers=projection.raw.atomic_numbers_chart,
        l_feature=projection.z_l,
    )
    if mode == 'none':
        f_new = state.f.detach().clone()
        v_new = state.v.detach().clone()
    elif mode == 'final_projection_only':
        f_new = projection.z_frac.detach().clone()
        v_new = state.v.detach().clone()
    elif mode == 'velocity_tangent_only':
        v_new = tangent_velocity
        f_new = kinetic_position_update(state.f, v_new, state.dt)
    elif mode == 'coord_projection_plus_tangent':
        v_new = tangent_velocity
        f_new = kinetic_position_update(projection.z_frac, v_new, state.dt)
    elif mode == 'normal_force_plus_tangent':
        forced = apply_full_space_force(state.v, residual=-residual, step_size=eta, mean_free=False)
        v_new, _projector2, _ = tangent_project_velocity(forced, J=J, metric=metric, damping=FIXED_CFG.projector_damping, mean_free=True)
        f_new = kinetic_position_update(state.f, v_new, state.dt)
    elif mode == 'reduced_space':
        omega = reduced_chart_velocity(state.v, J=J, metric=metric, damping=FIXED_CFG.projector_damping)
        residual_theta = reduced_chart_velocity(residual, J=J, metric=metric, damping=FIXED_CFG.projector_damping)
        omega = apply_reduced_space_force(omega, residual=-residual_theta, step_size=eta)
        v_new = lift_reduced_chart_velocity(omega, J=J, shape=state.v.shape)
        v_new, mean_norm_before = center_velocity(v_new)
        f_new = kinetic_position_update(state.f, v_new, state.dt)
    else:
        raise ValueError(f'unsupported repair mode: {mode}')
    repaired = KLDMState(
        f=f_new,
        v=v_new,
        l=state.l.detach().clone(),
        k=state.k.detach().clone(),
        h=state.h.detach().clone(),
        t=state.t,
        dt=state.dt,
        graph_idx0=state.graph_idx0,
        full_state=state.full_state,
        full_times=state.full_times,
    )
    dist_final, final_projection, final_J = template_distance(case, fixture, repaired.f, k=repaired.k, atomic_numbers=repaired.h, l_feature=repaired.l)
    info = {
        'dist_before': float(torch.linalg.norm(residual.reshape(-1)).detach().item()),
        'dist_after_projection': dist_after_projection,
        'dist_after': dist_final,
        'velocity_norm_before': graph_velocity_norm(state.v),
        'velocity_norm_after': graph_velocity_norm(v_new),
        'mean_norm_before': mean_norm_before,
        'mean_norm_after': mean_norm(v_new),
        'tangent_residual': projector.residual_norm(v_new),
        'projection_objective': float(projection.objective),
        'projection_energy_final': float(final_projection.objective),
        'rank_J': int(projector.rank),
        'cond_J': float(projector.condition_number),
        'min_pair_distance': min_pair_distance_from_arrays(repaired.f, repaired.l, int(repaired.f.shape[0])),
        'volume_ratio': volume_from_l(repaired.l, int(repaired.f.shape[0])) / max(volume_from_l(fixture.chart_l, int(fixture.chart_num_atoms)), 1.0e-8),
    }
    return repaired, projection, J, final_projection, final_J, info


def __run_short_chain_base(case: GraphCase, fixture: FixedTemplateFixture, *, mode: str, start_step: int, end_step: int, metric_name: str = 'fractional', eta: float = 0.05, final_projection: bool = False):
    set_seed(SAMPLE_SEED)
    full_state = runner.model._prepare_csp_sampling(
        batch=batch,
        n_steps=CAPTURE_N_STEPS,
        t_start=float(facit_cfg.get('t_start', 1.0)),
        t_final=float(facit_cfg.get('t_final', 1e-3)),
    )
    dists = []
    projection_fail_count = 0
    start_time = float('nan')
    end_time = float('nan')
    for step_idx, times in enumerate(iter_sampling_times(batch=full_state['batch'], grid=full_state['sampling_time_grid']), start=1):
        if step_idx < start_step:
            full_state = _native_reverse_step_full_state(full_state, times)
            continue
        if step_idx > end_step:
            break
        if math.isnan(start_time):
            start_time = float(times.now.graph[case.graph_idx0].detach().reshape(-1)[0].item())
        full_state = _native_reverse_step_full_state(full_state, times)
        state = graph_state_from_full(full_state, case, times)
        if mode != 'none':
            try:
                repaired, _proj, _J, _proj_final, _J_final, _info = one_step_repair(case, fixture, state, mode=mode, metric_name=metric_name, eta=eta)
                start, end = graph_slice(case.graph_idx0)
                if int(repaired.f.shape[0]) == int(end - start):
                    full_state['f_t'][start:end] = repaired.f.to(device=full_state['f_t'].device, dtype=full_state['f_t'].dtype)
                    full_state['v_t'][start:end] = repaired.v.to(device=full_state['v_t'].device, dtype=full_state['v_t'].dtype)
            except Exception:
                projection_fail_count += 1
        state = graph_state_from_full(full_state, case, times)
        dist_now, _proj_now, _J_now = template_distance(case, fixture, state.f, k=state.k, atomic_numbers=state.h, l_feature=state.l)
        dists.append(dist_now)
        end_time = float(times.now.graph[case.graph_idx0].detach().reshape(-1)[0].item())
    final_state = graph_state_from_full(full_state, case, times)
    if final_projection:
        proj_end, _J_end = project_state_to_fixture(case, fixture, final_state.f, k=final_state.k, atomic_numbers=final_state.h, l_feature=final_state.l)
        final_state = KLDMState(
            f=proj_end.z_frac.detach().clone(),
            v=final_state.v.detach().clone(),
            l=final_state.l.detach().clone(),
            k=proj_end.z_k.detach().clone(),
            h=proj_end.raw.atomic_numbers_chart.detach().clone(),
            t=final_state.t,
            dt=final_state.dt,
            graph_idx0=final_state.graph_idx0,
            full_state=final_state.full_state,
            full_times=final_state.full_times,
        )
        end_dist, _proj_end2, _J_end2 = template_distance(case, fixture, final_state.f, k=final_state.k, atomic_numbers=final_state.h, l_feature=final_state.l)
        dists.append(end_dist)
    evaluation = evaluate_arrays(case, final_state.f, final_state.l, final_state.h)
    score_norms = score_norms_from_modified_state(case, final_state, f=final_state.f, v=final_state.v, l=final_state.l)
    return {
        'dist_initial': dists[0] if dists else float('nan'),
        'dist_final': dists[-1] if dists else float('nan'),
        'projection_fail_count': int(projection_fail_count),
        'start_time': start_time,
        'end_time': end_time,
        'velocity_norm_max': float(max([0.0] + [graph_velocity_norm(final_state.v)])),
        'score_norm_max': float(max(score_norms['score_v_norm'], score_norms['pred_l_norm'])) if score_norms['finite_ok'] else float('nan'),
        **evaluation,
    }


def run_short_chain(case: GraphCase, fixture: FixedTemplateFixture, *, mode: str, start_step: int, end_step: int, metric_name: str = 'fractional', eta: float = 0.05, final_projection: bool = False):
    if ORACLE_W_MODE and fixture.uses_oracle_chart and fixture.chart_num_atoms != fixture.raw_num_atoms:
        dists = []
        projection_fail_count = 0
        start_time = float('nan')
        end_time = float('nan')
        final_state = None
        for step_idx in range(int(start_step), int(end_step) + 1):
            raw_state = sample_kldm_state_for_graph_at_step(case, capture_step=step_idx, n_steps=CAPTURE_N_STEPS)
            state = _state_to_fixture_chart(case, fixture, raw_state)
            if math.isnan(start_time):
                start_time = float(state.t)
            if mode != 'none':
                try:
                    repaired, _proj, _J, _proj_final, _J_final, _info = one_step_repair(case, fixture, state, mode=mode, metric_name=metric_name, eta=eta)
                    state = repaired
                except Exception:
                    projection_fail_count += 1
            dist_now, _proj_now, _J_now = template_distance(case, fixture, state.f, k=state.k, atomic_numbers=state.h, l_feature=state.l)
            dists.append(dist_now)
            end_time = float(state.t)
            final_state = state
        if final_state is None:
            final_state = gt_state_for_graph(case)
        if final_projection:
            proj_end, _J_end = project_state_to_fixture(case, fixture, final_state.f, k=final_state.k, atomic_numbers=final_state.h, l_feature=final_state.l)
            final_state = KLDMState(
                f=proj_end.z_frac.detach().clone(),
                v=final_state.v.detach().clone(),
                l=final_state.l.detach().clone(),
                k=proj_end.z_k.detach().clone(),
                h=proj_end.raw.atomic_numbers_chart.detach().clone(),
                t=final_state.t,
                dt=final_state.dt,
                graph_idx0=final_state.graph_idx0,
                full_state=final_state.full_state,
                full_times=final_state.full_times,
            )
            end_dist, _proj_end2, _J_end2 = template_distance(case, fixture, final_state.f, k=final_state.k, atomic_numbers=final_state.h, l_feature=final_state.l)
            dists.append(end_dist)
        evaluation = evaluate_arrays(case, final_state.f, final_state.l, final_state.h)
        score_norms = score_norms_from_modified_state(case, final_state, f=final_state.f, v=final_state.v, l=final_state.l)
        return {
            'dist_initial': dists[0] if dists else float('nan'),
            'dist_final': dists[-1] if dists else float('nan'),
            'projection_fail_count': int(projection_fail_count),
            'start_time': start_time,
            'end_time': end_time,
            'velocity_norm_max': float(graph_velocity_norm(final_state.v)),
            'score_norm_max': float(max(score_norms['score_v_norm'], score_norms['pred_l_norm'])) if score_norms['finite_ok'] else float('nan'),
            'oracle_shadow_chain': True,
            **evaluation,
        }
    return __run_short_chain_base(
        case,
        fixture,
        mode=mode,
        start_step=start_step,
        end_step=end_step,
        metric_name=metric_name,
        eta=eta,
        final_projection=final_projection,
    )


In [5]:
fixtures = {}
rows = []
for case in GRAPH_CASES:
    try:
        fixture = build_fixed_template_fixture(case)
        fixtures[case.graph_id] = fixture
        rows.append({
            'test': '0_setup_and_fixture',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'num_atoms': int(case.gt_frac.shape[0]),
            'num_species': len(case.composition),
            'template_labels': fixture.template_labels,
            'template_multiplicities': fixture.template_multiplicities,
            'template_dofs': fixture.template_dofs,
            'template_total_dof': fixture.template_total_dof,
            'projection_objective': fixture.projection_objective,
            'requested_sg_match': fixture.requested_sg_match,
            'composition_match': fixture.composition_match,
            'chart_num_atoms': fixture.chart_num_atoms,
            'target_representation': fixture.target_representation_name,
            'anchor_representation': fixture.anchor_representation_name,
            'uses_oracle_chart': fixture.uses_oracle_chart,
            'PASS': bool(fixture.composition_match and fixture.requested_sg_match),
            'status': 'PASS' if bool(fixture.composition_match and fixture.requested_sg_match) else 'FAIL',
        })
    except Exception as exc:
        rows.append(error_row('0_setup_and_fixture', case.graph_id, exc, 'FIXTURE_BUILD_ERROR', space_group=case.requested_sg))
df_0 = dataframe_result('0_setup_and_fixture', rows)
display(df_0)
print_group_pass(df_0, 'graph')


,test,graph,space_group,num_atoms,num_species,template_labels,template_multiplicities,template_dofs,template_total_dof,projection_objective,requested_sg_match,composition_match,chart_num_atoms,target_representation,anchor_representation,uses_oracle_chart,PASS,status
0,0_setup_and_fixture,1,227,6,2,"70@8a,77@16d","8,16","0,0",0,1.442308e-02,True,True,24,expanded,expanded,True,True,PASS
1,0_setup_and_fixture,2,4,16,3,"26@2a,26@2a,26@2a,27@2a,27@2a,27@2a,5@2a,5@2a","2,2,2,2,2,2,2,2","3,3,3,3,3,3,3,3",24,5.482579e-10,True,True,16,expanded,expanded,True,True,PASS
2,0_setup_and_fixture,3,194,8,2,"72@2a,22@6h","2,6","0,1",1,2.238027e-02,True,True,8,raw_requested_expanded,raw_requested_expanded,True,True,PASS


,graph,PASS,status
0,1,True,PASS
1,2,True,PASS
2,3,True,PASS


In [6]:
# Gates 1-3: fixed-template validity, projection, idempotence
rows_1 = []
rows_2 = []
rows_3 = []
for case in GRAPH_CASES:
    fixture = fixtures.get(case.graph_id)
    if fixture is None:
        continue
    try:
        theta_rand = sample_random_free_vars(
            fixture.state.template,
            device=fixture.theta.device,
            dtype=fixture.theta.dtype,
        )
        mat = materialize_template(theta_rand, fixture.state, tau=fixture.tau)
        structure = _build_vanilla_structure(
            frac_coords=mat.frac_coords,
            atomic_numbers=mat.atomic_numbers,
            cell_matrix=mat.cell,
        )
        expected_atomic_numbers = fixture.chart_atomic_numbers if (ORACLE_W_MODE and fixture.uses_oracle_chart) else case.atomic_numbers
        expected_shape_ok = bool(torch.isfinite(mat.frac_coords).all() and int(mat.frac_coords.shape[0]) == int(expected_atomic_numbers.shape[0]))
        validation = validate_requested_space_group(
            structure=structure,
            requested_space_group=case.requested_sg,
            expected_atomic_numbers=expected_atomic_numbers,
        )
        rows_1.append({
            'test': '1_fixed_template_validity',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'template_labels': fixture.template_labels,
            'theta_dim': fixture.template_total_dof,
            'num_atoms_materialized': int(mat.frac_coords.shape[0]),
            'composition_ok': bool(validation.composition_match),
            'requested_sg_match': bool(validation.requested_space_group_match),
            'site_shape_ok': expected_shape_ok,
            'PASS': bool(validation.composition_match and expected_shape_ok),
            'status': 'PASS' if bool(validation.composition_match and expected_shape_ok) else 'FAIL',
        })
    except Exception as exc:
        rows_1.append(error_row('1_fixed_template_validity', case.graph_id, exc, 'FIXED_TEMPLATE_VALIDITY_ERROR'))
    try:
        source_states = {
            'gt': gt_state_for_graph(case),
            'random': random_fractional_state(case),
            'kldm': sample_kldm_state_for_graph_at_step(case, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS),
        }
        for source_name, source_state in source_states.items():
            anchor_residual = float(torch.linalg.norm(wrap_residual(source_state.f, fixture.z_frac).reshape(-1)).detach().item())
            projection, J = project_state_to_fixture(case, fixture, source_state.f, k=source_state.k, atomic_numbers=source_state.h, l_feature=source_state.l)
            after_residual = float(torch.linalg.norm(wrap_residual(source_state.f, projection.z_frac).reshape(-1)).detach().item())
            rows_2.append({
                'test': '2_fixed_template_projection',
                'graph': case.graph_id,
                'source': source_name,
                'coord_residual_before': anchor_residual,
                'coord_residual_after': after_residual,
                'projection_energy': projection.objective,
                'num_ssvd_iters': getattr(projection.raw, 'ssvd_steps', None),
                'line_search_fail_count': getattr(projection.raw, 'ssvd_line_search_failures', None),
                'rank_J': int(torch.linalg.matrix_rank(J).detach().item()) if J.numel() else 0,
                'cond_J': float(tangent_projector(J, damping=FIXED_CFG.projector_damping).condition_number),
                'PASS': bool(np.isfinite(after_residual) and np.isfinite(projection.objective)),
                'status': 'PASS' if bool(np.isfinite(after_residual) and np.isfinite(projection.objective)) else 'FAIL',
            })
        idempotent, _J = project_state_to_fixture(
            case,
            fixture,
            fixture.z_frac,
            k=fixture.z_k,
            use_self_target=True,
            atomic_numbers=fixture.chart_atomic_numbers if (ORACLE_W_MODE and fixture.uses_oracle_chart) else case.atomic_numbers,
            l_feature=fixture.chart_l if (ORACLE_W_MODE and fixture.uses_oracle_chart) else fixture.z_l,
        )
        theta_error = float(torch.linalg.norm((idempotent.theta - fixture.theta).reshape(-1)).detach().item()) if fixture.theta.numel() else 0.0
        idempotence_residual = float(torch.linalg.norm(wrap_residual(fixture.z_frac, idempotent.z_frac).reshape(-1)).detach().item())
        rows_3.append({
            'test': '3_projection_idempotence',
            'graph': case.graph_id,
            'theta_dim': fixture.template_total_dof,
            'idempotence_residual': idempotence_residual,
            'theta_error': theta_error,
            'PASS': bool(idempotence_residual <= 1.0e-5),
            'status': 'PASS' if bool(idempotence_residual <= 1.0e-5) else 'FAIL',
        })
    except Exception as exc:
        rows_2.append(error_row('2_fixed_template_projection', case.graph_id, exc, 'FIXED_TEMPLATE_PROJECTION_ERROR'))
        rows_3.append(error_row('3_projection_idempotence', case.graph_id, exc, 'FIXED_TEMPLATE_IDEMPOTENCE_ERROR'))

df_1 = dataframe_result('1_fixed_template_validity', rows_1)
df_2 = dataframe_result('2_fixed_template_projection', rows_2)
df_3 = dataframe_result('3_projection_idempotence', rows_3)
display(df_1)
display(df_2)
display(df_3)


,test,graph,space_group,template_labels,theta_dim,num_atoms_materialized,composition_ok,requested_sg_match,site_shape_ok,PASS,status
0,1_fixed_template_validity,1,227,"70@8a,77@16d",0,24,True,True,True,True,PASS
1,1_fixed_template_validity,2,4,"26@2a,26@2a,26@2a,27@2a,27@2a,27@2a,5@2a,5@2a",24,16,True,True,True,True,PASS
2,1_fixed_template_validity,3,194,"72@2a,22@6h",1,8,True,True,True,True,PASS


,test,graph,source,coord_residual_before,coord_residual_after,projection_energy,num_ssvd_iters,line_search_fail_count,rank_J,cond_J,PASS,status
0,2_fixed_template_projection,1,gt,2.031010,2.031010,1.442308e-02,16,0,0,inf,True,PASS
1,2_fixed_template_projection,1,random,2.031010,2.031010,1.442308e-02,16,0,0,inf,True,PASS
2,2_fixed_template_projection,1,kldm,2.031010,2.031010,1.442308e-02,16,0,0,inf,True,PASS
3,2_fixed_template_projection,2,gt,2.145301,2.038591,1.441782e-11,16,0,24,1.0,True,PASS
4,2_fixed_template_projection,2,random,1.900236,1.599405,4.593837e-02,16,0,24,1.0,True,PASS
5,2_fixed_template_projection,2,kldm,1.734834,2.029815,3.644753e-02,16,0,24,1.0,True,PASS
6,2_fixed_template_projection,3,gt,1.272167,1.272167,2.238027e-02,16,0,1,1.0,True,PASS
7,2_fixed_template_projection,3,random,1.422282,1.424800,7.409696e-02,16,0,1,1.0,True,PASS
8,2_fixed_template_projection,3,kldm,1.271025,1.289249,7.549366e-02,16,0,1,1.0,True,PASS


,test,graph,theta_dim,idempotence_residual,theta_error,PASS,status
0,3_projection_idempotence,1,0,0.0,0.0,True,PASS
1,3_projection_idempotence,2,24,0.0,0.0,True,PASS
2,3_projection_idempotence,3,1,0.0,0.0,True,PASS


In [7]:
# Gates 4-7: Jacobian, tangent projector, zero-DOF handling, mean-free compatibility
rows_4 = []
rows_5 = []
rows_6 = []
rows_7 = []
for case in GRAPH_CASES:
    fixture = fixtures.get(case.graph_id)
    if fixture is None:
        continue
    try:
        theta_dim = int(fixture.theta.numel())
        if theta_dim == 0:
            for eps in FD_EPS_VALUES:
                rows_4.append({
                    'test': '4_jacobian_fd_check',
                    'graph': case.graph_id,
                    'theta_dim': 0,
                    'epsilon': eps,
                    'fd_error': 0.0,
                    'relative_fd_error': 0.0,
                    'PASS': True,
                    'status': 'PASS',
                })
        else:
            direction = torch.randn(theta_dim, device=fixture.theta.device, dtype=fixture.theta.dtype)
            direction = direction / direction.norm().clamp_min(1.0e-12)
            for eps in FD_EPS_VALUES:
                fd_error, rel_error = finite_difference_jacobian_error(
                    theta=fixture.theta,
                    direction=direction,
                    epsilon=eps,
                    template_state=fixture.state,
                    tau=fixture.tau,
                    jacobian=fixture.J,
                )
                rows_4.append({
                    'test': '4_jacobian_fd_check',
                    'graph': case.graph_id,
                    'theta_dim': theta_dim,
                    'epsilon': eps,
                    'fd_error': fd_error,
                    'relative_fd_error': rel_error,
                    'PASS': bool(np.isfinite(rel_error)),
                    'status': 'PASS' if bool(np.isfinite(rel_error)) else 'FAIL',
                })
        state = sample_kldm_state_for_graph_at_step(case, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS)
        for metric_name in METRIC_MODES:
            metric = metric_tensor(metric_name, fixture.z_l.to(device=fixture.z_frac.device, dtype=fixture.z_frac.dtype), int(fixture.z_frac.shape[0]))
            projected_v, projector, _mean_norm_before = tangent_project_velocity(state.v, J=fixture.J, metric=metric, damping=FIXED_CFG.projector_damping, mean_free=False)
            J_cols = [projector.project_flat(fixture.J[:, idx]) for idx in range(fixture.J.shape[1])] if fixture.J.shape[1] > 0 else []
            projector_preserves_J = float(torch.linalg.norm(torch.stack(J_cols, dim=1) - fixture.J).detach().item()) if J_cols else 0.0
            rows_5.append({
                'test': '5_tangent_projector_algebra',
                'graph': case.graph_id,
                'metric': metric_name,
                'rank_J': projector.rank,
                'cond_J': projector.condition_number,
                'projector_idempotence_error': projector_idempotence_error(projector, state.v),
                'projector_preserves_J_error': projector_preserves_J,
                'tangent_residual': projector.residual_norm(projected_v),
                'PASS': bool(projector.residual_norm(projected_v) <= 1.0e-5 or projector.rank == 0),
                'status': 'PASS' if bool(projector.residual_norm(projected_v) <= 1.0e-5 or projector.rank == 0) else 'FAIL',
            })
        site_ranks = site_jacobian_block_ranks(fixture.J, template_state=fixture.state)
        site_slices = []
        start = 0
        for site in fixture.state.template.site_templates:
            width = int(site.multiplicity)
            site_slices.append(slice(start, start + width))
            start += width
        tangent_v, projector, _ = tangent_project_velocity(state.v, J=fixture.J, metric=metric_tensor('fractional', fixture.z_l.to(device=fixture.z_frac.device, dtype=fixture.z_frac.dtype), int(fixture.z_frac.shape[0])), damping=FIXED_CFG.projector_damping, mean_free=False)
        for site, site_slice, site_rank in zip(fixture.state.template.site_templates, site_slices, site_ranks, strict=True):
            rows_6.append({
                'test': '6_rank_and_zero_dof_handling',
                'graph': case.graph_id,
                'site_label': site.label,
                'multiplicity': int(site.multiplicity),
                'site_dof': int(site.dof),
                'jacobian_block_rank': int(site_rank),
                'velocity_norm_before': float(torch.linalg.norm(state.v[site_slice].reshape(-1)).detach().item()),
                'velocity_norm_after': float(torch.linalg.norm(tangent_v[site_slice].reshape(-1)).detach().item()),
                'PASS': True,
                'status': 'PASS',
            })
        tangent_centered, projector, mean_norm_before = tangent_project_velocity(state.v, J=fixture.J, metric=metric_tensor('fractional', fixture.z_l.to(device=fixture.z_frac.device, dtype=fixture.z_frac.dtype), int(fixture.z_frac.shape[0])), damping=FIXED_CFG.projector_damping, mean_free=True)
        rows_7.append({
            'test': '7_mean_free_compatibility',
            'graph': case.graph_id,
            'mean_norm_before': mean_norm_before,
            'mean_norm_after': mean_norm(tangent_centered),
            'tangent_residual_after_centering': tangent_residual_after_centering(tangent_centered, J=fixture.J, metric=metric_tensor('fractional', fixture.z_l.to(device=fixture.z_frac.device, dtype=fixture.z_frac.dtype), int(fixture.z_frac.shape[0])), damping=FIXED_CFG.projector_damping),
            'PASS': bool(mean_norm(tangent_centered) <= 1.0e-6),
            'status': 'PASS' if bool(mean_norm(tangent_centered) <= 1.0e-6) else 'FAIL',
        })
    except Exception as exc:
        rows_4.append(error_row('4_jacobian_fd_check', case.graph_id, exc, 'JACOBIAN_CHECK_ERROR'))
        rows_5.append(error_row('5_tangent_projector_algebra', case.graph_id, exc, 'TANGENT_PROJECTOR_ERROR'))
        rows_6.append(error_row('6_rank_and_zero_dof_handling', case.graph_id, exc, 'ZERO_DOF_ERROR'))
        rows_7.append(error_row('7_mean_free_compatibility', case.graph_id, exc, 'MEAN_FREE_ERROR'))

df_4 = dataframe_result('4_jacobian_fd_check', rows_4)
df_5 = dataframe_result('5_tangent_projector_algebra', rows_5)
df_6 = dataframe_result('6_rank_and_zero_dof_handling', rows_6)
df_7 = dataframe_result('7_mean_free_compatibility', rows_7)
display(df_4)
display(df_5)
display(df_6)
display(df_7)


,test,graph,theta_dim,epsilon,fd_error,relative_fd_error,PASS,status
0,4_jacobian_fd_check,1,0,0.0100,0.000000,0.000000e+00,True,PASS
1,4_jacobian_fd_check,1,0,0.0010,0.000000,0.000000e+00,True,PASS
2,4_jacobian_fd_check,1,0,0.0001,0.000000,0.000000e+00,True,PASS
3,4_jacobian_fd_check,2,24,0.0100,0.000015,1.060696e-05,True,PASS
4,4_jacobian_fd_check,2,24,0.0010,0.000148,1.049456e-04,True,PASS
5,4_jacobian_fd_check,2,24,0.0001,0.001412,9.983187e-04,True,PASS
6,4_jacobian_fd_check,3,1,0.0100,0.000005,9.536752e-07,True,PASS
7,4_jacobian_fd_check,3,1,0.0010,0.000063,1.293437e-05,True,PASS
8,4_jacobian_fd_check,3,1,0.0001,0.000813,1.659118e-04,True,PASS


,test,graph,metric,rank_J,cond_J,projector_idempotence_error,projector_preserves_J_error,tangent_residual,PASS,status
0,5_tangent_projector_algebra,1,fractional,0,inf,0.000000e+00,0.000000e+00,0.000000e+00,True,PASS
1,5_tangent_projector_algebra,1,cartesian,0,inf,0.000000e+00,0.000000e+00,0.000000e+00,True,PASS
2,5_tangent_projector_algebra,2,fractional,24,1.000000,3.044528e-07,3.303625e-06,3.044528e-07,True,PASS
3,5_tangent_projector_algebra,2,cartesian,24,2.243737,4.712161e-08,0.000000e+00,4.712161e-08,True,PASS
4,5_tangent_projector_algebra,3,fractional,1,1.000000,1.710949e-09,2.920019e-07,1.710949e-09,True,PASS
5,5_tangent_projector_algebra,3,cartesian,1,1.000000,0.000000e+00,5.840039e-07,0.000000e+00,True,PASS


,test,graph,site_label,multiplicity,site_dof,jacobian_block_rank,velocity_norm_before,velocity_norm_after,PASS,status
0,6_rank_and_zero_dof_handling,1,8a,8,0,0,0.406063,0.000000,True,PASS
1,6_rank_and_zero_dof_handling,1,16d,16,0,0,0.521536,0.000000,True,PASS
2,6_rank_and_zero_dof_handling,2,2a,2,3,3,0.261380,0.108951,True,PASS
3,6_rank_and_zero_dof_handling,2,2a,2,3,3,0.246876,0.154609,True,PASS
4,6_rank_and_zero_dof_handling,2,2a,2,3,3,0.526082,0.199881,True,PASS
5,6_rank_and_zero_dof_handling,2,2a,2,3,3,0.387997,0.303040,True,PASS
6,6_rank_and_zero_dof_handling,2,2a,2,3,3,0.385634,0.232055,True,PASS
7,6_rank_and_zero_dof_handling,2,2a,2,3,3,0.481500,0.327160,True,PASS
8,6_rank_and_zero_dof_handling,2,2a,2,3,3,0.226507,0.213055,True,PASS
9,6_rank_and_zero_dof_handling,2,2a,2,3,3,0.251209,0.145410,True,PASS


,test,graph,mean_norm_before,mean_norm_after,tangent_residual_after_centering,PASS,status
0,7_mean_free_compatibility,1,0.000000e+00,0.000000e+00,0.000000e+00,True,PASS
1,7_mean_free_compatibility,2,1.862645e-09,1.862645e-09,3.044528e-07,True,PASS
2,7_mean_free_compatibility,3,0.000000e+00,0.000000e+00,1.710949e-09,True,PASS


In [8]:
# Gates 8-10: local tangent preservation and off-manifold corrections
rows_8 = []
rows_9 = []
rows_10 = []
for case in GRAPH_CASES:
    fixture = fixtures.get(case.graph_id)
    if fixture is None:
        continue
    try:
        state = sample_kldm_state_for_graph_at_step(case, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS)
        h = abs(float(state.dt)) if np.isfinite(state.dt) and abs(float(state.dt)) > 0 else 1.0 / max(1, CAPTURE_N_STEPS)
        raw_velocity = state.v.detach().clone()
        tangent_velocity, projector, _ = tangent_project_velocity(raw_velocity, J=fixture.J, metric=metric_tensor('fractional', fixture.z_l.to(device=fixture.z_frac.device, dtype=fixture.z_frac.dtype), int(fixture.z_frac.shape[0])), damping=FIXED_CFG.projector_damping, mean_free=True)
        f_raw = kinetic_position_update(fixture.z_frac, raw_velocity, h)
        f_tan = kinetic_position_update(fixture.z_frac, tangent_velocity, h)
        dist_raw, _proj_raw, _J_raw = template_distance(case, fixture, f_raw, k=fixture.z_k)
        dist_tan, _proj_tan, _J_tan = template_distance(case, fixture, f_tan, k=fixture.z_k)
        rows_8.append({
            'test': '8_one_step_tangent_preservation',
            'graph': case.graph_id,
            'h': h,
            'dist_after_raw_velocity': dist_raw,
            'dist_after_tangent_velocity': dist_tan,
            'improvement_ratio': dist_raw / max(dist_tan, 1.0e-12),
            'PASS': bool(dist_tan <= dist_raw + EPS_PASS),
            'status': 'PASS' if bool(dist_tan <= dist_raw + EPS_PASS) else 'FAIL',
        })
        dist_initial, projection, J = template_distance(case, fixture, state.f, k=state.k)
        metric = metric_tensor('fractional', projection.z_l.to(device=state.f.device, dtype=state.f.dtype), int(state.f.shape[0]))
        residual = wrap_residual(state.f, projection.z_frac)
        tangent_only, _proj_a, _m_a = tangent_project_velocity(state.v, J=J, metric=metric, damping=FIXED_CFG.projector_damping, mean_free=True)
        normal_only = apply_full_space_force(state.v, residual=-residual, step_size=0.05, mean_free=True)
        normal_then_tangent_forced = apply_full_space_force(state.v, residual=-residual, step_size=0.05, mean_free=False)
        normal_then_tangent, _proj_c, _m_c = tangent_project_velocity(normal_then_tangent_forced, J=J, metric=metric, damping=FIXED_CFG.projector_damping, mean_free=True)
        modes = {
            'tangent_only': tangent_only,
            'normal_only': normal_only,
            'normal_then_tangent': normal_then_tangent,
        }
        distances = {}
        for mode_name, velocity in modes.items():
            f_next = kinetic_position_update(state.f, velocity, h)
            distances[mode_name] = template_distance(case, fixture, f_next, k=state.k)[0]
        rows_9.append({
            'test': '9_off_manifold_correction',
            'graph': case.graph_id,
            'eta': 0.05,
            'h': h,
            'dist_initial': dist_initial,
            'dist_tangent_only': distances['tangent_only'],
            'dist_normal_only': distances['normal_only'],
            'dist_normal_then_tangent': distances['normal_then_tangent'],
            'best_mode': min(distances, key=distances.get),
            'PASS': bool(min(distances.values()) <= dist_initial + EPS_PASS),
            'status': 'PASS' if bool(min(distances.values()) <= dist_initial + EPS_PASS) else 'FAIL',
        })
        repaired, projection0, J0, projection1, J1, info = one_step_repair(case, fixture, state, mode='coord_projection_plus_tangent', metric_name='fractional', eta=0.05)
        rows_10.append({
            'test': '10_projection_plus_tangent_velocity',
            'graph': case.graph_id,
            'dist_before': info['dist_before'],
            'dist_after_projection': float(torch.linalg.norm(wrap_residual(state.f, projection0.z_frac).reshape(-1)).detach().item()),
            'dist_after_one_step': info['dist_after'],
            'velocity_norm_before': info['velocity_norm_before'],
            'velocity_norm_after': info['velocity_norm_after'],
            'tangent_residual': info['tangent_residual'],
            'PASS': bool(info['dist_after'] <= info['dist_before'] + EPS_PASS),
            'status': 'PASS' if bool(info['dist_after'] <= info['dist_before'] + EPS_PASS) else 'FAIL',
        })
    except Exception as exc:
        rows_8.append(error_row('8_one_step_tangent_preservation', case.graph_id, exc, 'TANGENT_PRESERVATION_ERROR'))
        rows_9.append(error_row('9_off_manifold_correction', case.graph_id, exc, 'OFF_MANIFOLD_CORRECTION_ERROR'))
        rows_10.append(error_row('10_projection_plus_tangent_velocity', case.graph_id, exc, 'PROJECTION_PLUS_TANGENT_ERROR'))

df_8 = dataframe_result('8_one_step_tangent_preservation', rows_8)
df_9 = dataframe_result('9_off_manifold_correction', rows_9)
df_10 = dataframe_result('10_projection_plus_tangent_velocity', rows_10)
display(df_8)
display(df_9)
display(df_10)


,test,graph,h,dist_after_raw_velocity,dist_after_tangent_velocity,improvement_ratio,PASS,status
0,8_one_step_tangent_preservation,1,0.024975,2.031010,2.031010,1.000000,True,PASS
1,8_one_step_tangent_preservation,2,0.024975,1.331133,1.212543,1.097802,True,PASS
2,8_one_step_tangent_preservation,3,0.024975,1.242226,1.242239,0.999990,False,FAIL


,test,graph,eta,h,dist_initial,dist_tangent_only,dist_normal_only,dist_normal_then_tangent,best_mode,PASS,status
0,9_off_manifold_correction,1,0.05,0.024975,2.031010,2.031010,2.031010,2.031010,tangent_only,True,PASS
1,9_off_manifold_correction,2,0.05,0.024975,1.781539,1.785302,1.785008,1.785008,normal_then_tangent,False,FAIL
2,9_off_manifold_correction,3,0.05,0.024975,1.297406,1.297377,1.242153,1.297358,normal_only,True,PASS


,test,graph,dist_before,dist_after_projection,dist_after_one_step,velocity_norm_before,velocity_norm_after,tangent_residual,PASS,status
0,10_projection_plus_tangent_velocity,1,2.031010,2.031010,2.031010,0.660974,0.000000,0.000000e+00,True,PASS
1,10_projection_plus_tangent_velocity,2,2.029815,2.029815,1.487533,1.025445,0.628474,3.044528e-07,True,PASS
2,10_projection_plus_tangent_velocity,3,1.289249,1.289249,1.242028,0.621598,0.009476,1.710949e-09,True,PASS


In [9]:
# Gates 11-12: score compatibility and one KLDM reverse step with repair
rows_11 = []
rows_12 = []
for case in GRAPH_CASES:
    fixture = fixtures.get(case.graph_id)
    if fixture is None:
        continue
    try:
        state = sample_kldm_state_for_graph_at_step(case, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS)
        projection, J = project_state_to_fixture(case, fixture, state.f, k=state.k, atomic_numbers=state.h, l_feature=state.l)
        tangent_v, projector, _ = tangent_project_velocity(state.v, J=J, metric=metric_tensor('fractional', projection.z_l.to(device=state.f.device, dtype=state.f.dtype), int(state.f.shape[0])), damping=FIXED_CFG.projector_damping, mean_free=True)
        original_norms = score_norms_from_modified_state(case, state, f=state.f, v=state.v, l=state.l)
        variants = {
            'original': (state.f, state.v, state.l),
            'coord_projected': (projection.z_frac, state.v, state.l),
            'coord_plus_tangent_velocity': (projection.z_frac, tangent_v, state.l),
        }
        for variant_name, (f_var, v_var, l_var) in variants.items():
            norms = score_norms_from_modified_state(case, state, f=f_var, v=v_var, l=l_var)
            ratio = float(norms['score_v_norm'] / max(original_norms['score_v_norm'], 1.0e-12)) if np.isfinite(original_norms['score_v_norm']) else float('nan')
            rows_11.append({
                'test': '11_score_compatibility_after_projection',
                'graph': case.graph_id,
                'state_type': variant_name,
                **norms,
                'score_ratio_vs_original': ratio,
                'PASS': bool(norms['finite_ok'] and (not np.isfinite(ratio) or ratio <= 3.0 + EPS_PASS)),
                'status': 'PASS' if bool(norms['finite_ok'] and (not np.isfinite(ratio) or ratio <= 3.0 + EPS_PASS)) else 'FAIL',
            })
        full_state, times, _ = capture_batch_kldm_state(seed=SAMPLE_SEED, n_steps=CAPTURE_N_STEPS, capture_step=CAPTURE_STEP)
        step_state = clone_full_state(full_state)
        step_state = _native_reverse_step_full_state(step_state, times)
        post_kldm = graph_state_from_full(step_state, case, times)
        dist_before, _proj_before, _J_before = template_distance(case, fixture, state.f, k=state.k, atomic_numbers=state.h, l_feature=state.l)
        dist_after_kldm, _proj_after, _J_after = template_distance(case, fixture, post_kldm.f, k=post_kldm.k, atomic_numbers=post_kldm.h, l_feature=post_kldm.l)
        for mode in ['none', 'final_projection_only', 'velocity_tangent_only', 'coord_projection_plus_tangent', 'normal_force_plus_tangent', 'reduced_space']:
            repaired, projection0, J0, projection1, J1, info = one_step_repair(case, fixture, post_kldm, mode=mode, metric_name='fractional', eta=0.05)
            rows_12.append({
                'test': '12_one_reverse_step_with_fixed_projection',
                'graph': case.graph_id,
                'mode': mode,
                'time': post_kldm.t,
                'dist_before': dist_before,
                'dist_after_kldm': dist_after_kldm,
                'dist_after_repair': info['dist_after'],
                'velocity_norm': info['velocity_norm_after'],
                'score_norm': rows_11[-1]['score_v_norm'] if rows_11 else float('nan'),
                'min_pair_distance': info['min_pair_distance'],
                'volume_ratio': info['volume_ratio'],
                'PASS': bool(info['dist_after'] <= dist_after_kldm + EPS_PASS),
                'status': 'PASS' if bool(info['dist_after'] <= dist_after_kldm + EPS_PASS) else 'FAIL',
            })
    except Exception as exc:
        rows_11.append(error_row('11_score_compatibility_after_projection', case.graph_id, exc, 'SCORE_COMPATIBILITY_ERROR'))
        rows_12.append(error_row('12_one_reverse_step_with_fixed_projection', case.graph_id, exc, 'ONE_STEP_REPAIR_ERROR'))

df_11 = dataframe_result('11_score_compatibility_after_projection', rows_11)
df_12 = dataframe_result('12_one_reverse_step_with_fixed_projection', rows_12)
display(df_11)
display(df_12)


,test,graph,state_type,pred_v_norm,score_v_norm,pred_l_norm,finite_ok,score_ratio_vs_original,PASS,status
0,11_score_compatibility_after_projection,1,original,NaN,NaN,NaN,False,NaN,False,FAIL
1,11_score_compatibility_after_projection,1,coord_projected,NaN,NaN,NaN,False,NaN,False,FAIL
2,11_score_compatibility_after_projection,1,coord_plus_tangent_velocity,NaN,NaN,NaN,False,NaN,False,FAIL
3,11_score_compatibility_after_projection,2,original,8.167268,175.552673,1.983873,True,1.000000,True,PASS
4,11_score_compatibility_after_projection,2,coord_projected,20.863327,344.786591,1.768799,True,1.964006,True,PASS
5,11_score_compatibility_after_projection,2,coord_plus_tangent_velocity,21.729053,340.248108,1.775504,True,1.938154,True,PASS
6,11_score_compatibility_after_projection,3,original,6.480755,139.710861,1.634689,True,1.000000,True,PASS
7,11_score_compatibility_after_projection,3,coord_projected,9.157736,121.277565,2.108468,True,0.868061,True,PASS
8,11_score_compatibility_after_projection,3,coord_plus_tangent_velocity,4.432411,67.283134,2.119561,True,0.481588,True,PASS


,test,graph,mode,time,dist_before,dist_after_kldm,dist_after_repair,velocity_norm,score_norm,min_pair_distance,volume_ratio,PASS,status
0,12_one_reverse_step_with_fixed_projection,1,none,0.125875,2.031010,2.031010,2.031010,0.444667,NaN,2.664643,1.000000,True,PASS
1,12_one_reverse_step_with_fixed_projection,1,final_projection_only,0.125875,2.031010,2.031010,2.031010,0.444667,NaN,2.664643,1.000000,True,PASS
2,12_one_reverse_step_with_fixed_projection,1,velocity_tangent_only,0.125875,2.031010,2.031010,2.031010,0.000000,NaN,2.664643,1.000000,True,PASS
3,12_one_reverse_step_with_fixed_projection,1,coord_projection_plus_tangent,0.125875,2.031010,2.031010,2.031010,0.000000,NaN,2.664643,1.000000,True,PASS
4,12_one_reverse_step_with_fixed_projection,1,normal_force_plus_tangent,0.125875,2.031010,2.031010,2.031010,0.000000,NaN,2.664643,1.000000,True,PASS
5,12_one_reverse_step_with_fixed_projection,1,reduced_space,0.125875,2.031010,2.031010,2.031010,0.000000,NaN,2.664643,1.000000,True,PASS
6,12_one_reverse_step_with_fixed_projection,2,none,0.125875,1.781539,1.790267,1.790267,0.913096,340.248108,1.386247,0.714134,True,PASS
7,12_one_reverse_step_with_fixed_projection,2,final_projection_only,0.125875,1.781539,1.790267,1.490559,0.913096,340.248108,0.930854,0.714134,True,PASS
8,12_one_reverse_step_with_fixed_projection,2,velocity_tangent_only,0.125875,1.781539,1.790267,1.794802,0.599338,340.248108,1.412339,0.714134,False,FAIL
9,12_one_reverse_step_with_fixed_projection,2,coord_projection_plus_tangent,0.125875,1.781539,1.790267,1.485841,0.599338,340.248108,0.942900,0.714134,True,PASS


In [10]:
# Gates 13-16: synthetic kinetic CASAL, dual placement, reduced/full-space correction, metric choice
rows_13 = []
rows_14 = []
rows_15 = []
rows_16 = []
for case in GRAPH_CASES:
    fixture = fixtures.get(case.graph_id)
    if fixture is None:
        continue
    try:
        state = sample_kldm_state_for_graph_at_step(case, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS)
        projection0, J0 = project_state_to_fixture(case, fixture, state.f, k=state.k)
        for use_tangent_projection in [False, True]:
            x = state.f.detach().clone()
            v = state.v.detach().clone()
            z = projection0.z_frac.detach().clone()
            mu_f = torch.zeros_like(x)
            dists = []
            for _step in range(8):
                residual = wrap_residual(x, z) + mu_f
                noise = 0.01 * torch.randn_like(v)
                v = v - 0.05 * residual - 0.02 * v + noise
                if use_tangent_projection:
                    J_local = compute_template_jacobian(projection0.theta, fixture.state, tau=fixture.tau)
                    v, _proj_local, _ = tangent_project_velocity(v, J=J_local, metric=metric_tensor('fractional', fixture.z_l.to(device=fixture.z_frac.device, dtype=fixture.z_frac.dtype), int(fixture.z_frac.shape[0])), damping=FIXED_CFG.projector_damping, mean_free=True)
                x = kinetic_position_update(x, v, max(abs(state.dt), 1.0 / CAPTURE_N_STEPS))
                proj_x, _J_x = project_state_to_fixture(case, fixture, x, k=state.k)
                z = proj_x.z_frac.detach().clone()
                mu_f = mu_f + 0.01 * wrap_residual(x, z)
                dists.append(float(torch.linalg.norm(wrap_residual(x, z).reshape(-1)).detach().item()))
            rows_13.append({
                'test': '13_synthetic_casal_without_kldm_score',
                'graph': case.graph_id,
                'eta_v': 0.05,
                'eta_d': 0.02,
                'eta_mu': 0.01,
                'sigma': 0.01,
                'use_tangent_projection': use_tangent_projection,
                'dist_initial': float(torch.linalg.norm(wrap_residual(state.f, projection0.z_frac).reshape(-1)).detach().item()),
                'dist_final': dists[-1] if dists else float('nan'),
                'mu_norm_final': float(torch.linalg.norm(mu_f.reshape(-1)).detach().item()),
                'velocity_norm_final': float(torch.linalg.norm(v.reshape(-1)).detach().item()),
                'oscillation_score': float(np.std(np.diff(dists))) if len(dists) > 1 else 0.0,
                'PASS': bool((dists[-1] if dists else float('inf')) <= (dists[0] if dists else float('inf')) + EPS_PASS),
                'status': 'PASS' if bool((dists[-1] if dists else float('inf')) <= (dists[0] if dists else float('inf')) + EPS_PASS) else 'FAIL',
            })
        for policy in ['every_step', 'projection_success_only', 'reduced_dual']:
            x = state.f.detach().clone()
            v = state.v.detach().clone()
            proj = projection0
            mu_f = torch.zeros_like(x)
            mu_theta = torch.zeros(int(J0.shape[1]), device=x.device, dtype=x.dtype)
            dual_clip_count = 0
            for _step in range(6):
                residual = wrap_residual(x, proj.z_frac)
                if policy == 'reduced_dual' and mu_theta.numel() > 0:
                    mu_full = lift_reduced_chart_velocity(mu_theta, J=J0, shape=tuple(x.shape))
                else:
                    mu_full = mu_f
                v = v - 0.05 * (residual + mu_full)
                x = kinetic_position_update(x, v, max(abs(state.dt), 1.0 / CAPTURE_N_STEPS))
                proj_new, J_new = project_state_to_fixture(case, fixture, x, k=state.k)
                if policy == 'every_step':
                    mu_f = mu_f + 0.01 * wrap_residual(x, proj_new.z_frac)
                elif policy == 'projection_success_only':
                    mu_f = mu_f + 0.01 * wrap_residual(x, proj_new.z_frac)
                else:
                    metric = metric_tensor('fractional', proj_new.z_l.to(device=x.device, dtype=x.dtype), int(x.shape[0]))
                    mu_theta = mu_theta + 0.01 * (J_new.transpose(0, 1) @ metric @ wrap_residual(x, proj_new.z_frac).reshape(-1))
                dual_clip_count += int(float(torch.linalg.norm(mu_f.reshape(-1)).detach().item()) > 1.0)
                proj = proj_new
                J0 = J_new
            rows_14.append({
                'test': '14_dual_update_placement',
                'graph': case.graph_id,
                'policy': policy,
                'dist_final': float(torch.linalg.norm(wrap_residual(x, proj.z_frac).reshape(-1)).detach().item()),
                'mu_norm_final': float(torch.linalg.norm(mu_f.reshape(-1)).detach().item()) if policy != 'reduced_dual' else float(torch.linalg.norm(mu_theta.reshape(-1)).detach().item()),
                'dual_clip_fraction': dual_clip_count / 6.0,
                'velocity_norm_final': float(torch.linalg.norm(v.reshape(-1)).detach().item()),
                'PASS': True,
                'status': 'PASS',
            })
        for mode in ['full_space', 'reduced_space']:
            projection, J = project_state_to_fixture(case, fixture, state.f, k=state.k)
            metric = metric_tensor('fractional', projection.z_l.to(device=state.f.device, dtype=state.f.dtype), int(state.f.shape[0]))
            residual = wrap_residual(state.f, projection.z_frac)
            if mode == 'full_space':
                v_mode = apply_full_space_force(state.v, residual=-residual, step_size=0.05, mean_free=True)
            else:
                omega = reduced_chart_velocity(state.v, J=J, metric=metric, damping=FIXED_CFG.projector_damping)
                residual_theta = reduced_chart_velocity(residual, J=J, metric=metric, damping=FIXED_CFG.projector_damping)
                omega = apply_reduced_space_force(omega, residual=-residual_theta, step_size=0.05)
                v_mode = lift_reduced_chart_velocity(omega, J=J, shape=state.v.shape)
                v_mode, _ = center_velocity(v_mode)
            f_mode = kinetic_position_update(state.f, v_mode, max(abs(state.dt), 1.0 / CAPTURE_N_STEPS))
            dist_final, proj_final, J_final = template_distance(case, fixture, f_mode, k=state.k)
            rows_15.append({
                'test': '15_full_space_vs_reduced_space_correction',
                'graph': case.graph_id,
                'mode': mode,
                'dist_initial': float(torch.linalg.norm(residual.reshape(-1)).detach().item()),
                'dist_final': dist_final,
                'velocity_norm': float(torch.linalg.norm(v_mode.reshape(-1)).detach().item()),
                'tangent_residual': tangent_projector(J_final, metric=metric, damping=FIXED_CFG.projector_damping).residual_norm(v_mode),
                'PASS': bool(dist_final <= float(torch.linalg.norm(residual.reshape(-1)).detach().item()) + EPS_PASS),
                'status': 'PASS' if bool(dist_final <= float(torch.linalg.norm(residual.reshape(-1)).detach().item()) + EPS_PASS) else 'FAIL',
            })
        for metric_name in METRIC_MODES:
            repaired, projection0, J0, projection1, J1, info = one_step_repair(case, fixture, state, mode='coord_projection_plus_tangent', metric_name=metric_name, eta=0.05)
            evaluation = evaluate_arrays(case, repaired.f, repaired.l, repaired.h)
            rows_16.append({
                'test': '16_metric_choice',
                'graph': case.graph_id,
                'metric': metric_name,
                'dist_frac_final': info['dist_after'],
                'dist_cart_final': cart_distance_to_projection(repaired.f, projection1, int(repaired.f.shape[0])),
                'rmse': evaluation['rmse'],
                'match': evaluation['match'],
                'velocity_norm': info['velocity_norm_after'],
                'PASS': True,
                'status': 'PASS',
            })
    except Exception as exc:
        rows_13.append(error_row('13_synthetic_casal_without_kldm_score', case.graph_id, exc, 'SYNTHETIC_CASAL_ERROR'))
        rows_14.append(error_row('14_dual_update_placement', case.graph_id, exc, 'DUAL_PLACEMENT_ERROR'))
        rows_15.append(error_row('15_full_space_vs_reduced_space_correction', case.graph_id, exc, 'REDUCED_SPACE_ERROR'))
        rows_16.append(error_row('16_metric_choice', case.graph_id, exc, 'METRIC_CHOICE_ERROR'))

df_13 = dataframe_result('13_synthetic_casal_without_kldm_score', rows_13)
df_14 = dataframe_result('14_dual_update_placement', rows_14)
df_15 = dataframe_result('15_full_space_vs_reduced_space_correction', rows_15)
df_16 = dataframe_result('16_metric_choice', rows_16)
display(df_13)
display(df_14)
display(df_15)
display(df_16)


,test,graph,eta_v,eta_d,eta_mu,sigma,use_tangent_projection,dist_initial,dist_final,mu_norm_final,velocity_norm_final,oscillation_score,PASS,status
0,13_synthetic_casal_without_kldm_score,1,0.05,0.02,0.01,0.01,False,2.031010,2.137152,0.166108,1.091819,0.005316,False,FAIL
1,13_synthetic_casal_without_kldm_score,1,0.05,0.02,0.01,0.01,True,2.031010,2.031010,0.162481,0.000000,0.000000,True,PASS
2,13_synthetic_casal_without_kldm_score,2,0.05,0.02,0.01,0.01,False,2.029815,2.093331,0.147400,1.125036,0.001723,False,FAIL
3,13_synthetic_casal_without_kldm_score,2,0.05,0.02,0.01,0.01,True,2.029815,2.028100,0.159952,0.532567,0.000755,True,PASS
4,13_synthetic_casal_without_kldm_score,3,0.05,0.02,0.01,0.01,False,1.289249,1.294381,0.094886,0.663719,0.002068,False,FAIL
5,13_synthetic_casal_without_kldm_score,3,0.05,0.02,0.01,0.01,True,1.289249,1.269968,0.090553,0.046390,0.006845,True,PASS


,test,graph,policy,dist_final,mu_norm_final,dual_clip_fraction,velocity_norm_final,PASS,status
0,14_dual_update_placement,1,every_step,2.095712,0.123629,0.0,0.957467,True,PASS
1,14_dual_update_placement,1,projection_success_only,2.095712,0.123629,0.0,0.957467,True,PASS
2,14_dual_update_placement,1,reduced_dual,2.094817,0.000000,0.0,0.946354,True,PASS
3,14_dual_update_placement,2,every_step,2.076425,0.114859,0.0,1.217787,True,PASS
4,14_dual_update_placement,2,projection_success_only,2.076425,0.114859,0.0,1.217787,True,PASS
5,14_dual_update_placement,2,reduced_dual,2.076017,0.063634,0.0,1.210496,True,PASS
6,14_dual_update_placement,3,every_step,1.279577,0.073009,0.0,0.651265,True,PASS
7,14_dual_update_placement,3,projection_success_only,1.279577,0.073009,0.0,0.651265,True,PASS
8,14_dual_update_placement,3,reduced_dual,1.279139,0.024769,0.0,0.649587,True,PASS


,test,graph,mode,dist_initial,dist_final,velocity_norm,tangent_residual,PASS,status
0,15_full_space_vs_reduced_space_correction,1,full_space,2.031010,2.031010,0.675163,6.751626e-01,True,PASS
1,15_full_space_vs_reduced_space_correction,1,reduced_space,2.031010,2.031010,0.000000,0.000000e+00,True,PASS
2,15_full_space_vs_reduced_space_correction,2,full_space,2.029815,1.785012,1.039337,8.314400e-01,True,PASS
3,15_full_space_vs_reduced_space_correction,2,reduced_space,2.029815,1.785012,0.623641,3.040080e-07,True,PASS
4,15_full_space_vs_reduced_space_correction,3,full_space,1.289249,1.242153,0.608291,6.081002e-01,True,PASS
5,15_full_space_vs_reduced_space_correction,3,reduced_space,1.289249,1.297360,0.015235,0.000000e+00,False,FAIL


,test,graph,metric,dist_frac_final,dist_cart_final,rmse,match,velocity_norm,PASS,status
0,16_metric_choice,1,fractional,2.031010,0.000000,6.502575e-16,True,0.000000,True,PASS
1,16_metric_choice,1,cartesian,2.031010,0.000000,6.502575e-16,True,0.000000,True,PASS
2,16_metric_choice,2,fractional,1.487533,12.827325,NaN,False,0.628474,True,PASS
3,16_metric_choice,2,cartesian,1.487533,12.827325,NaN,False,0.628474,True,PASS
4,16_metric_choice,3,fractional,1.242028,1.933214,NaN,False,0.009476,True,PASS
5,16_metric_choice,3,cartesian,1.242066,1.921696,NaN,False,0.174031,True,PASS


In [11]:
# Gates 17-20: mini-chain, full-chain, template sensitivity, chart-state score compatibility
rows_17 = []
rows_18 = []
rows_19 = []
rows_20 = []
for case in GRAPH_CASES:
    fixture = fixtures.get(case.graph_id)
    if fixture is None:
        continue
    try:
        for mode in ['none', 'coord_projection_plus_tangent', 'normal_force_plus_tangent', 'reduced_space']:
            summary = run_short_chain(case, fixture, mode=mode, start_step=LATE_START_STEP, end_step=min(CAPTURE_N_STEPS, LATE_START_STEP + MINI_CHAIN_STEPS - 1), metric_name='fractional', eta=0.05, final_projection=False)
            rows_17.append({
                'test': '17_fixed_template_kldm_mini_chain',
                'graph': case.graph_id,
                'mode': mode,
                'n_steps': MINI_CHAIN_STEPS,
                **summary,
                'PASS': True,
                'status': 'PASS',
            })
        for method, mode, final_projection in [
            ('KLDM_baseline', 'none', False),
            ('KLDM_final_fixed_projection', 'none', True),
            ('KLDM_final_projection_plus_velocity_reset', 'velocity_tangent_only', True),
            ('late_fixed_template_repair', 'coord_projection_plus_tangent', False),
            ('late_fixed_template_kinetic_casal', 'normal_force_plus_tangent', False),
            ('late_fixed_template_kinetic_casal_plus_final_projection', 'normal_force_plus_tangent', True),
        ]:
            summary = run_short_chain(case, fixture, mode=mode, start_step=1, end_step=FULL_CHAIN_STEPS, metric_name='fractional', eta=0.05, final_projection=final_projection)
            rows_18.append({
                'test': '18_full_reverse_chain_with_fixed_template',
                'graph': case.graph_id,
                'method': method,
                **summary,
                'valid_structure': summary['valid'],
                'dist_to_template_final': summary['dist_final'],
                'status': 'PASS' if bool(summary['valid']) else 'DIAG',
                'PASS': bool(summary['valid']),
            })
        kldm_state = sample_kldm_state_for_graph_at_step(case, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS)
        target_cell = cell_from_l(kldm_state.l, int(kldm_state.f.shape[0])).to(device=kldm_state.f.device, dtype=kldm_state.f.dtype)
        candidate_states = select_requested_template_states(
            frac_coords=kldm_state.f,
            atomic_numbers=kldm_state.h,
            cell_matrix=target_cell,
            space_group_number=case.requested_sg,
            standardization='conventional',
            symprec=1e-2,
            angle_tolerance=5.0,
            max_templates=max(TEMPLATE_TOP_K, FIXTURE_MAX_TEMPLATES),
            template_eval_limit=max(TEMPLATE_TOP_K, FIXTURE_TEMPLATE_EVAL_LIMIT),
            optimization_steps=FIXTURE_OPT_STEPS,
            learning_rate=FIXTURE_LR,
            coord_weight=1.0,
            lattice_weight=0.0,
            pairdist_weight=0.0,
            steric_weight=0.0,
            volume_weight=0.0,
            k6_weight=0.0,
            freeze_lattice_free_vars=True,
            quick_templates=False,
            top_k=TEMPLATE_TOP_K,
            template_prior=template_prior,
            template_prior_weight=0.0,
        )
        for rank_idx, candidate in enumerate(candidate_states, start=1):
            projection = project_to_fixed_template(
                f_frac=kldm_state.f,
                atomic_numbers=kldm_state.h,
                template_state=candidate,
                target_k=kldm_state.k,
                tau0=fixture.tau,
                config=FIXED_CFG,
            )
            evaluation = evaluate_arrays(case, projection.z_frac, projection.z_l, projection.raw.atomic_numbers_chart)
            rows_19.append({
                'test': '19_template_sensitivity',
                'graph': case.graph_id,
                'template_rank': rank_idx,
                'template_labels': template_labels(candidate.template),
                'dist_final': float(torch.linalg.norm(wrap_residual(kldm_state.f, projection.z_frac).reshape(-1)).detach().item()),
                'projection_energy': projection.objective,
                'match': evaluation['match'],
                'rmse': evaluation['rmse'],
                'requested_sg_match': evaluation['requested_sg_match'],
                'PASS': True,
                'status': 'PASS',
            })
        theta_noise_values = [0.01, 0.05]
        omega_noise_values = [0.01, 0.05]
        base_state = sample_kldm_state_for_graph_at_step(case, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS)
        for theta_noise in theta_noise_values:
            for omega_noise in omega_noise_values:
                theta = fixture.theta.detach().clone()
                if theta.numel() > 0:
                    theta = theta + theta_noise * torch.randn_like(theta)
                mat = materialize_template(theta, fixture.state, tau=fixture.tau)
                omega = omega_noise * torch.randn(int(fixture.J.shape[1]), device=fixture.J.device, dtype=fixture.J.dtype)
                v = lift_reduced_chart_velocity(omega, J=fixture.J, shape=tuple(fixture.z_frac.shape))
                v, _ = center_velocity(v)
                norms = score_norms_from_modified_state(case, base_state, f=mat.frac_coords, v=v, l=fixture.chart_l)
                evaluation = evaluate_arrays(case, mat.frac_coords, fixture.chart_l, mat.atomic_numbers)
                rows_20.append({
                    'test': '20_diffcsp_style_fixed_chart_compatibility',
                    'graph': case.graph_id,
                    'theta_noise': theta_noise,
                    'omega_noise': omega_noise,
                    'score_norm': norms['score_v_norm'],
                    'dist_to_template': float(torch.linalg.norm(wrap_residual(mat.frac_coords, fixture.z_frac).reshape(-1)).detach().item()),
                    'rmse': evaluation['rmse'],
                    'finite_ok': norms['finite_ok'],
                    'PASS': bool(norms['finite_ok']),
                    'status': 'PASS' if bool(norms['finite_ok']) else 'FAIL',
                })
    except Exception as exc:
        rows_17.append(error_row('17_fixed_template_kldm_mini_chain', case.graph_id, exc, 'MINI_CHAIN_ERROR'))
        rows_18.append(error_row('18_full_reverse_chain_with_fixed_template', case.graph_id, exc, 'FULL_CHAIN_ERROR'))
        rows_19.append(error_row('19_template_sensitivity', case.graph_id, exc, 'TEMPLATE_SENSITIVITY_ERROR'))
        rows_20.append(error_row('20_diffcsp_style_fixed_chart_compatibility', case.graph_id, exc, 'CHART_COMPATIBILITY_ERROR'))

df_17 = dataframe_result('17_fixed_template_kldm_mini_chain', rows_17)
df_18 = dataframe_result('18_full_reverse_chain_with_fixed_template', rows_18)
df_19 = dataframe_result('19_template_sensitivity', rows_19)
df_20 = dataframe_result('20_diffcsp_style_fixed_chart_compatibility', rows_20)
display(df_17)
display(df_18)
display(df_19)
display(df_20)


,test,graph,mode,n_steps,dist_initial,dist_final,projection_fail_count,start_time,end_time,velocity_norm_max,...,volume,lattice_lengths_mae,lattice_angles_mae,validity_reason,PASS,status,error_type,error_message,traceback_tail,failure_category
0,17_fixed_template_kldm_mini_chain,1,none,12.0,2.031010,2.031010,0.0,0.225775,0.025975,0.765733,...,428.106810,2.207462,29.999996,ok,True,PASS,NaN,NaN,NaN,NaN
1,17_fixed_template_kldm_mini_chain,1,coord_projection_plus_tangent,12.0,2.031010,2.031010,0.0,0.225775,0.025975,0.000000,...,428.106810,2.207462,29.999996,ok,True,PASS,NaN,NaN,NaN,NaN
2,17_fixed_template_kldm_mini_chain,1,normal_force_plus_tangent,12.0,2.031010,2.031010,0.0,0.225775,0.025975,0.000000,...,428.106810,2.207462,29.999996,ok,True,PASS,NaN,NaN,NaN,NaN
3,17_fixed_template_kldm_mini_chain,1,reduced_space,12.0,2.031010,2.031010,0.0,0.225775,0.025975,0.000000,...,428.106810,2.207462,29.999996,ok,True,PASS,NaN,NaN,NaN,NaN
4,17_fixed_template_kldm_mini_chain,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,False,ERROR,SymmetryUndeterminedError,,pymatgen.symmetry.analyzer.SymmetryUndetermine...,MINI_CHAIN_ERROR
5,17_fixed_template_kldm_mini_chain,2,none,12.0,1.806974,1.804860,0.0,0.225775,0.025975,0.399183,...,160.697622,0.639133,2.850339,ok,True,PASS,NaN,NaN,NaN,NaN
6,17_fixed_template_kldm_mini_chain,2,coord_projection_plus_tangent,12.0,1.399945,1.398781,0.0,0.225775,0.025975,0.458877,...,191.214470,0.663367,5.824839,ok,True,PASS,NaN,NaN,NaN,NaN
7,17_fixed_template_kldm_mini_chain,2,normal_force_plus_tangent,12.0,1.752645,1.808733,0.0,0.225775,0.025975,0.374965,...,157.981589,0.583188,3.239271,ok,True,PASS,NaN,NaN,NaN,NaN
8,17_fixed_template_kldm_mini_chain,2,reduced_space,12.0,1.752645,1.808733,0.0,0.225775,0.025975,0.374964,...,157.981567,0.583189,3.239263,ok,True,PASS,NaN,NaN,NaN,NaN
9,17_fixed_template_kldm_mini_chain,3,none,12.0,1.320823,1.251158,0.0,0.225775,0.025975,0.465015,...,170.502831,0.660519,8.072505,ok,True,PASS,NaN,NaN,NaN,NaN


,test,graph,PASS,status,error_type,error_message,traceback_tail,failure_category,method,dist_initial,...,standardized_frac_rmse,composition_match,requested_sg_match,min_pair_distance,volume,lattice_lengths_mae,lattice_angles_mae,validity_reason,valid_structure,dist_to_template_final
0,18_full_reverse_chain_with_fixed_template,1,False,ERROR,SymmetryUndeterminedError,,pymatgen.symmetry.analyzer.SymmetryUndetermine...,FULL_CHAIN_ERROR,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,18_full_reverse_chain_with_fixed_template,2,True,PASS,NaN,NaN,NaN,NaN,KLDM_baseline,1.981701,...,NaN,True,False,1.840461,160.697622,0.639133,2.850339,ok,True,1.804860
2,18_full_reverse_chain_with_fixed_template,2,False,DIAG,NaN,NaN,NaN,NaN,KLDM_final_fixed_projection,1.981701,...,NaN,True,False,0.927431,160.697622,0.639133,2.850339,close_contacts,False,1.484533
3,18_full_reverse_chain_with_fixed_template,2,False,DIAG,NaN,NaN,NaN,NaN,KLDM_final_projection_plus_velocity_reset,1.977312,...,NaN,True,False,0.461331,159.461595,0.592615,2.269643,close_contacts,False,1.480262
4,18_full_reverse_chain_with_fixed_template,2,True,PASS,NaN,NaN,NaN,NaN,late_fixed_template_repair,1.273007,...,0.156337,True,False,1.230510,257.316746,1.046569,6.660347,ok,True,1.957101
5,18_full_reverse_chain_with_fixed_template,2,True,PASS,NaN,NaN,NaN,NaN,late_fixed_template_kinetic_casal,1.977793,...,0.189182,True,False,1.845228,175.764803,0.369551,13.909037,ok,True,2.011747
6,18_full_reverse_chain_with_fixed_template,2,True,PASS,NaN,NaN,NaN,NaN,late_fixed_template_kinetic_casal_plus_final_p...,1.977793,...,0.181340,True,False,1.087614,175.764803,0.369551,13.909037,ok,True,1.656030
7,18_full_reverse_chain_with_fixed_template,3,True,PASS,NaN,NaN,NaN,NaN,KLDM_baseline,1.351760,...,NaN,True,False,2.525131,170.502831,0.660519,8.072505,ok,True,1.251158
8,18_full_reverse_chain_with_fixed_template,3,True,PASS,NaN,NaN,NaN,NaN,KLDM_final_fixed_projection,1.351760,...,0.182760,True,False,1.266188,170.502831,0.660519,8.072505,ok,True,1.247469
9,18_full_reverse_chain_with_fixed_template,3,False,DIAG,NaN,NaN,NaN,NaN,KLDM_final_projection_plus_velocity_reset,1.352541,...,NaN,True,False,0.302360,180.470855,1.532082,18.253639,close_contacts,False,1.245470


,test,graph,PASS,status,error_type,error_message,traceback_tail,failure_category,template_rank,template_labels,dist_final,projection_energy,match,rmse,requested_sg_match
0,19_template_sensitivity,1,False,ERROR,SymmetryUndeterminedError,,pymatgen.symmetry.analyzer.SymmetryUndetermine...,TEMPLATE_SENSITIVITY_ERROR,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,19_template_sensitivity,2,True,PASS,NaN,NaN,NaN,NaN,1.0,"26@2a,26@2a,26@2a,27@2a,27@2a,27@2a,5@2a,5@2a",2.172637,0.037481,False,NaN,True
2,19_template_sensitivity,3,True,PASS,NaN,NaN,NaN,NaN,1.0,"72@2a,22@6h",1.285203,0.090194,False,NaN,True
3,19_template_sensitivity,3,True,PASS,NaN,NaN,NaN,NaN,2.0,"72@2b,22@2d,22@2c,22@2a",1.627224,0.067398,False,NaN,False
4,19_template_sensitivity,3,True,PASS,NaN,NaN,NaN,NaN,3.0,"72@2a,22@2d,22@2c,22@2b",1.580647,0.070815,False,NaN,False


,test,graph,PASS,status,error_type,error_message,traceback_tail,failure_category,theta_noise,omega_noise,score_norm,dist_to_template,rmse,finite_ok
0,20_diffcsp_style_fixed_chart_compatibility,1,False,ERROR,SymmetryUndeterminedError,,pymatgen.symmetry.analyzer.SymmetryUndetermine...,CHART_COMPATIBILITY_ERROR,NaN,NaN,NaN,NaN,NaN,NaN
1,20_diffcsp_style_fixed_chart_compatibility,2,True,PASS,NaN,NaN,NaN,NaN,0.01,0.01,131.657883,0.077496,0.048018,True
2,20_diffcsp_style_fixed_chart_compatibility,2,True,PASS,NaN,NaN,NaN,NaN,0.01,0.05,194.341370,0.059377,0.036003,True
3,20_diffcsp_style_fixed_chart_compatibility,2,True,PASS,NaN,NaN,NaN,NaN,0.05,0.01,366.534882,0.333783,0.212324,True
4,20_diffcsp_style_fixed_chart_compatibility,2,True,PASS,NaN,NaN,NaN,NaN,0.05,0.05,385.029877,0.336430,0.216702,True
5,20_diffcsp_style_fixed_chart_compatibility,3,True,PASS,NaN,NaN,NaN,NaN,0.01,0.01,185.696899,0.004265,NaN,True
6,20_diffcsp_style_fixed_chart_compatibility,3,True,PASS,NaN,NaN,NaN,NaN,0.01,0.05,135.735229,0.011049,NaN,True
7,20_diffcsp_style_fixed_chart_compatibility,3,True,PASS,NaN,NaN,NaN,NaN,0.05,0.01,194.924881,0.002976,NaN,True
8,20_diffcsp_style_fixed_chart_compatibility,3,True,PASS,NaN,NaN,NaN,NaN,0.05,0.05,228.017487,0.068967,NaN,True


## Reading the notebook

A useful interpretation order is:

1. Gates 0-5 must be sane before any velocity story is believable.
2. Gates 8-10 tell us whether tangent projection is locally meaningful.
3. Gates 11-12 tell us whether KLDM even tolerates the repaired states.
4. Gates 17-18 decide whether fixed-template velocity repair beats `KLDM + final projection` on an actual CSP metric.
5. Gates 19-20 tell us whether the whole idea is brittle to template choice or chart-state score mismatch.

If Gate 18 cannot beat final fixed-template projection on any meaningful metric, then fixed-template velocity CASAL is probably not worth keeping as a sampler path.
